# Drive mount

In [ ]:
# libraries import
import pandas as pd
import os, re, glob, random, hashlib, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive, userdata

In [ ]:
!pip -q install pymupdf striprtf pytesseract umap-learn openai simplemma   # simplemma: B1 lemmas (was installed by NER cell A2)

In [ ]:
# Mount Drive and set all project paths
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)   # force: picks up newly created shortcuts

# Project folder: shortcut ("zástupce") in My Drive, found by part of its name
PROJECT_NAME_HINT = "TA2"
found = [p for pattern in ["*", "*/*", "*/*/*"]
         for p in Path("/content/drive/MyDrive").glob(pattern)
         if PROJECT_NAME_HINT.lower() in p.name.lower() and p.is_dir()]
assert found, f"No folder with '{PROJECT_NAME_HINT}' in My Drive. Runtime -> Disconnect and delete runtime, then rerun."
PROJECT_DIR = found[0]

# Programmes folder: matched case- and space-insensitively (real name is "Programy ")
RAW_DIR = next((p for p in PROJECT_DIR.iterdir()
                if p.is_dir() and p.name.strip().lower() == "programy"), None)
assert RAW_DIR, f"No programy folder in {PROJECT_DIR}. Contents: {[p.name for p in PROJECT_DIR.iterdir()]}"

# Superseded originals are kept as evidence in Programy/archive; the pipeline does not read them
ARCHIVE = RAW_DIR / "archive"; ARCHIVE.mkdir(exist_ok=True)
for name in ["spd_program.pdf", "voluntia_program.txt"]:
    if (RAW_DIR / name).exists(): (RAW_DIR / name).rename(ARCHIVE / name)
RAW_FILES = sorted(f for f in RAW_DIR.iterdir() if f.is_file())     # files only, no subfolders

SOURCES_CSV = PROJECT_DIR / "sources.csv"         # provenance table, maintained by hand
INTERIM_DIR = PROJECT_DIR / "data" / "interim"    # extraction cache
OCR_DIR     = INTERIM_DIR / "ocr"                 # OCR'd copies of scanned PDFs
CORPUS_DIR  = PROJECT_DIR / "data" / "corpus"     # final corpus_vN.jsonl
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR =", repr(str(PROJECT_DIR)))
print("RAW_DIR     =", repr(str(RAW_DIR)))
print("Files in RAW_DIR:", [f.name for f in RAW_FILES])

# ---- Embedding config (paths reuse the ones found above; no data is loaded here) ----
import re, glob, random, hashlib, time, json
import numpy as np
import pandas as pd
from google.colab import userdata

SEED = 42
random.seed(SEED); np.random.seed(SEED)

EMB_DIR     = PROJECT_DIR / "data" / "embeddings"
CLUSTER_DIR = PROJECT_DIR / "data" / "clusters"
EMB_DIR.mkdir(parents=True, exist_ok=True)
CLUSTER_DIR.mkdir(parents=True, exist_ok=True)

P = {
    "seed": SEED,
    "corpus_version": None,
    "embed_model": "qwen3-embedding:4b",
    "llm_model": "qwen3.6-35b",
    "base_url": "https://litellm.vse.cz/",
    "embed_batch": 32,
    "umap_dims_cluster": 10, "umap_n_neighbors": 15, "umap_min_dist": 0.0,
    "hdbscan_min_cluster_size": 10, "hdbscan_min_samples": None,
    "tfidf_min_df": 2, "svd_dims": 100, "centroid_reps": 5,
}

## Data check

Check what is really in the "Programy" folder, programmes are not always in pdf, this gives us the idea what file types are we working with.




In [ ]:
import fitz, hashlib, json, re
import pandas as pd

PARTY_ALIAS = {"voluntia_sections": "voluntia"}      # file stem -> party key

def party_of(path):
    # 'ano_program.pdf' -> 'ano'; 'spolu-portlik-program.txt' -> 'spolu-portlik'
    stem = re.sub(r"[_-]program.*$", "", path.stem)
    return PARTY_ALIAS.get(stem, stem)

rows = []
for p in RAW_FILES:
    pages = None
    if p.suffix.lower() == ".pdf":
        with fitz.open(p) as doc:
            pages = doc.page_count
    rows.append({
        "party": party_of(p),
        "file": p.name,
        "ext": p.suffix.lower(),
        "size_kb": round(p.stat().st_size / 1024),
        "sha256": hashlib.sha256(p.read_bytes()).hexdigest()[:16],  # short hash is enough for provenance
        "pdf_pages": pages,
    })
manifest = pd.DataFrame(rows)
manifest.to_csv(INTERIM_DIR / "manifest.csv", index=False)
display(manifest)

# One file per party is a rule of the pipeline: show the problem here, not three cells later
dup = manifest.groupby("party")["file"].nunique()
assert dup.eq(1).all(), f"More than one file for: {dup[dup > 1].to_dict()}. Move the superseded one to Programy/archive."

# Peek at the txt files and the json so we know their structure
for p in RAW_FILES:
    if p.suffix.lower() == ".txt":
        t = p.read_text(encoding="utf-8")
        print(f"\n--- {p.name}: {len(t)} chars ---\n{t[:300]}")
    if p.suffix.lower() == ".json":
        j = json.loads(p.read_text(encoding="utf-8"))
        print(f"\n--- {p.name}: top-level {type(j).__name__} ---\n{str(j)[:600]}")

# Your hand-made provenance table: check its columns
if SOURCES_CSV.exists():
    display(pd.read_csv(SOURCES_CSV).head(10))

**Note on raw counts.** These counts come from the raw extraction, before any cleaning.
Two parties are inflated by format, not by content:
- *voluntia*: the .txt file is RTF, so the count includes RTF control words.
- *pirati*: the JSON includes metadata (candidates, URLs, FAQ, situations), not only programme text.
Clean counts per party are reported from `corpus_v1` (phase 2).

## Data exploration




In [ ]:
def pages_pdf(p):
    # one (page_number, text) per page; page numbers start at 1
    with fitz.open(p) as doc:
        return [(i + 1, page.get_text("text")) for i, page in enumerate(doc)]

def strings_in(obj):
    # collect every string value in a nested json (keys are dropped)
    if isinstance(obj, str):
        yield obj
    elif isinstance(obj, dict):
        for v in obj.values(): yield from strings_in(v)
    elif isinstance(obj, list):
        for v in obj: yield from strings_in(v)

def pages_any(p):
    ext = p.suffix.lower()
    if ext == ".pdf":
        return pages_pdf(p)
    if ext == ".txt":    # no page info -> whole file is "page 1"
        return [(1, p.read_text(encoding="utf-8"))]
    if ext == ".json":   # raw layer: every string in the file; structure is read later
        return [(1, "\n\n".join(strings_in(json.loads(p.read_text(encoding="utf-8")))))]
    if ext == ".html":   # raw layer: visible text without menus/footers; structure is read later by read_spd
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(p.read_text(encoding="utf-8"), "html.parser")
        for t in soup(["script", "style", "nav", "footer"]): t.decompose()
        return [(1, soup.get_text("\n", strip=True))]
    raise ValueError(f"Unsupported file type: {p.name}")

def n_words(text):
    # whitespace split is enough for a size metric (no tokenizer needed)
    return len(text.split())

records = []
for p in RAW_FILES:
    for page_no, text in pages_any(p):
        records.append({"party": party_of(p), "file": p.name, "page": page_no,
                        "text": text, "n_chars": len(text), "n_words": n_words(text)})
pages = pd.DataFrame(records)
pages.to_csv(INTERIM_DIR / "pages.csv", index=False, encoding="utf-8")

summary = pages.groupby("party").agg(total_words=("n_words", "sum"),
                                     n_pages=("page", "count"))
summary["share_%"] = (100 * summary.total_words / summary.total_words.sum()).round(1)
display(summary.sort_values("total_words", ascending=False))

In [ ]:
import matplotlib.pyplot as plt

pages = pd.DataFrame(records)
pages.to_csv(INTERIM_DIR / "pages.csv", index=False, encoding="utf-8")
pages_before_ocr = pages.copy()   # snapshot of the text layer, taken here so a later rerun cannot overwrite it with OCR output             # snapshot, reused later for before/after comparison

s = pages_before_ocr.groupby("party").n_words.sum().sort_values(ascending=False)
share = (100 * s / s.sum()).round(1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(s.index, s.values, color="steelblue")
for i, (w, sh) in enumerate(zip(s.values, share.values)):
    ax.text(i, w, f"{sh}%", ha="center", va="bottom", fontsize=9)   # share of corpus on each bar
ax.set_title("Total words per party, before OCR (label = share of corpus)")
ax.set_ylabel("words")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(INTERIM_DIR / "explore_bar_before_ocr.png", dpi=150)
plt.show()



OCR

We need to install OCR, simply because some of the materials are printed websites which are renedered as scanned images and we have to extract the text using OCR.

In [ ]:
# install OCR
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ces
!pip -q install pytesseract
!tesseract --list-langs | grep ces      # must print: ces

This block check if we need OCR and for which files, so we do not have to check it manually, it also ensures, that we can upload other files and do not have to check, whether they need OCR treatment or not.

We decied to use this more complicated check, that verifies the

In [ ]:
# Rule: a PDF needs OCR if its text layer is missing or broken. Three simple tests per file.
CZ = set("ěščřžýáíéúůťďňĚŠČŘŽÝÁÍÉÚŮŤĎŇ")
if "source" not in pages:
    pages["source"] = "text_layer"             # marks where each page's text came from

rows, OCR_FILES = [], {}                        # OCR_FILES: file -> list of reasons
for p in sorted(RAW_DIR.glob("*.pdf")):
    g = pages[(pages.file == p.name) & (pages.source == "text_layer")]
    if g.empty:                                 # already OCR'd in an earlier run
        continue
    text = "".join(g.text)
    weak_share = (g.n_words < 10).mean()        # share of pages with almost no text
    diacritic = 100 * sum(c in CZ for c in text) / max(len(text), 1)
    broken = text.count("\ufffd")               # replacement chars = broken font encoding
    with fitz.open(p) as doc:
        img_pages = sum(len(pg.get_images()) > 0 for pg in doc)

    reasons = []
    if weak_share > 0.3:
        reasons.append(f"{weak_share:.0%} of pages have <10 words (no text layer; {img_pages}/{len(g)} pages contain images)")
    if broken > 0:
        reasons.append(f"{broken} replacement characters (broken font encoding)")
    if len(text) > 2000 and diacritic < 1:
        reasons.append(f"diacritic share {diacritic:.2f}% (Czech normally ~2-4%: diacritics lost)")
    if reasons:
        OCR_FILES[p.name] = reasons

    rows.append({"file": p.name, "pages": len(g), "median_words/page": g.n_words.median(),
                 "weak_pages_%": round(100 * weak_share), "pages_with_images": img_pages,
                 "replacement_chars": broken, "diacritic_%": round(diacritic, 2),
                 "needs_OCR": bool(reasons)})
display(pd.DataFrame(rows))

print("\nOCR needed for:" if OCR_FILES else "\nNo OCR needed.")
for f, rs in OCR_FILES.items():
    print(f"  {f}")
    for r in rs:
        print(f"     - {r}")

OCR deployment

In [ ]:
import pytesseract
from PIL import Image

def ocr_pdf(path):
    # render each page at 300 dpi (Tesseract's recommended resolution), OCR with the Czech model
    out = []
    with fitz.open(path) as doc:
        for i, page in enumerate(doc):
            pix = page.get_pixmap(dpi=300)
            img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
            out.append((i + 1, pytesseract.image_to_string(img, lang="ces")))
    return out

OCR_DIR.mkdir(parents=True, exist_ok=True)
new_rows = []
for fname in OCR_FILES:
    party = party_of(RAW_DIR / fname)
    cache = OCR_DIR / f"{party}_ocr.csv"
    if cache.exists():                          # OCR is slow: never redo it
        df = pd.read_csv(cache)
    else:
        df = pd.DataFrame(ocr_pdf(RAW_DIR / fname), columns=["page", "text"])
        df.to_csv(cache, index=False, encoding="utf-8")
    for _, r in df.fillna("").iterrows():
        new_rows.append({"party": party, "file": fname, "page": int(r.page), "text": r.text,
                         "n_chars": len(r.text), "n_words": n_words(r.text), "source": "ocr"})

if new_rows:
    old_words = pages[pages.file.isin(OCR_FILES)].groupby("party").n_words.sum()
    pages = (pd.concat([pages[~pages.file.isin(OCR_FILES)], pd.DataFrame(new_rows)])
               .sort_values(["party", "page"]).reset_index(drop=True))
    pages.to_csv(INTERIM_DIR / "pages.csv", index=False, encoding="utf-8")

    o = pages[pages.source == "ocr"]
    new_words = o.groupby("party").n_words.sum()
    display(pd.DataFrame({"words_before": old_words, "words_after": new_words}))
    print("Weak OCR pages (<10 words):", o[o.n_words < 10].groupby("party").page.apply(list).to_dict() or "none")
    print("Diacritic share %:", {p: round(100 * sum(c in CZ for c in "".join(g.text)) / max(len("".join(g.text)), 1), 2)
                                 for p, g in o.groupby("party")})
    for _, row in o.loc[o.n_words.nlargest(2).index].iterrows():     # read it yourself
        print(f"\n=== {row.party}, page {row.page} ({row.n_words} words) ===\n{row.text[:700]}")

In [ ]:
# Only parties with more than one page: TXT/JSON files count as a single "page", a box makes no sense
multi = pages.groupby("party").page.count()
multi = multi[multi > 1].index

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([pages[pages.party == p].n_words for p in multi],
           tick_labels=list(multi))                  # older matplotlib: use labels= instead of tick_labels=
ax.set_title("Words per page, after OCR (PDF parties only)")
ax.set_ylabel("words per page")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(INTERIM_DIR / "explore_box_after_ocr.png", dpi=150)
plt.show()

In [ ]:
# Save corpus
# One record per page, text exactly as extracte. Provenance: sha256 from the manifest.
manifest = pd.read_csv(INTERIM_DIR / "manifest.csv")
corpus = (pages.merge(manifest[["file", "sha256"]], on="file", how="left")
               .assign(doc_id=lambda d: d.party + "_" + d.page.astype(int).astype(str).str.zfill(3))
               [["doc_id", "party", "file", "page", "source", "sha256", "n_words", "n_chars", "text"]]
               .sort_values(["party", "page"]).reset_index(drop=True))

out = CORPUS_DIR / "corpus_v0_raw.jsonl"
corpus.to_json(out, orient="records", lines=True, force_ascii=False)   # force_ascii=False keeps Czech diacritics

# Check: reload gives the same table
back = pd.read_json(out, lines=True)
assert len(back) == len(corpus) and back.doc_id.is_unique
print(f"saved {len(corpus)} records -> {out.name}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

assert "pages_before_ocr" in globals(), "Re-run block C into `pages_before_ocr` first (the snapshot from before D2)."

before = pages_before_ocr.groupby("party").n_words.sum()
after  = pages.groupby("party").n_words.sum()
order  = after.sort_values(ascending=False).index

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Left: total words per party, before vs after OCR
x = np.arange(len(order))
ax[0].bar(x - 0.2, before.reindex(order), width=0.4, label="before OCR", color="lightgray")
ax[0].bar(x + 0.2, after.reindex(order),  width=0.4, label="after OCR",  color="steelblue")
ax[0].set_xticks(x); ax[0].set_xticklabels(order, rotation=45)
ax[0].set_ylabel("total words"); ax[0].set_title("Total words per party"); ax[0].legend()

# Right: words per page, PDF parties only (parties with more than one page), two boxes per party
multi = [p for p in order if (pages.party == p).sum() > 1]
for i, p in enumerate(multi):
    ax[1].boxplot(pages_before_ocr[pages_before_ocr.party == p].n_words, positions=[i - 0.2], widths=0.35,
                  patch_artist=True, boxprops=dict(facecolor="lightgray"), medianprops=dict(color="black"))
    ax[1].boxplot(pages[pages.party == p].n_words, positions=[i + 0.2], widths=0.35,
                  patch_artist=True, boxprops=dict(facecolor="steelblue"), medianprops=dict(color="black"))
ax[1].set_xticks(range(len(multi))); ax[1].set_xticklabels(multi, rotation=45)
ax[1].set_ylabel("words per page"); ax[1].set_title("Words per page, PDF parties (gray = before, blue = after OCR)")

plt.tight_layout()
plt.savefig(INTERIM_DIR / "explore_before_after_ocr.png", dpi=150)
plt.show()

In [ ]:
pd.set_option("display.max_colwidth", 70)
display(corpus.drop(columns="sha256").head(10))        # table header: first 10 records, text cut to 70 chars

print("\nRecords per party:")
display(corpus.groupby("party").agg(records=("doc_id", "count"), words=("n_words", "sum"),
                                    source=("source", "first")))

print("\nFirst 150 characters of the first record of each party:")
for _, r in corpus.groupby("party").head(1).iterrows():
    print(f"\n[{r.doc_id}] {r.text[:150]!r}")

# Preprocessing
Plan:





## Config and tunable parameters

In [ ]:
# E0 | Config for phase 2: input = corpus_v0_raw.jsonl from phase 1
from pathlib import Path

CORPUS_IN = CORPUS_DIR / 'corpus_v0_raw.jsonl'   # output of block 'Save corpus' above

# Raw corpus: one record per page (doc_id, party, file, page, source, sha256, n_words, n_chars, text)
corpus = pd.read_json(CORPUS_IN, lines=True)
assert corpus.groupby('party')['file'].nunique().eq(1).all()   # one source file per party

FILES = corpus.groupby('party')['file'].first().to_dict()      # party -> file name (no hardcoding)
SHA   = corpus.groupby('party')['sha256'].first().to_dict()    # provenance, copied into the output
OCR_PARTIES = set(corpus.loc[corpus.source == 'ocr', 'party'])  # spd, from block D1/D2
print(FILES, '\nOCR:', OCR_PARTIES)

# Tunable: change one value, re-run E3-G, compare the G table
P.update(HEAD_MAX_WORDS=20,   # R4: more words than this = body text
         H2_RATIO=1.2,        # PDF: font >= 1.2x body size -> sub-heading
         H1_RATIO=1.6,        # PDF: font >= 1.6x body size -> top heading
         MAX_WORDS=120)       # longer passages are split at sentence ends

Helper functions

In [ ]:
# E1 | Helpers: join wrapped lines, drop noise lines, unify bullets
UP = 'A-ZÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ'
def n_words(t): return len(t.split())

def join_lines(lines):
    """Join wrapped lines into one paragraph. 'civil-'+'ní' -> 'civilní', otherwise add a space."""
    out = ''
    for ln in lines:
        ln = ln.strip()
        if not ln: continue
        if re.search(r'\w-$', out) and ln[0].islower():   # word split by hyphen at line end
            out = out[:-1] + ln
        else:
            out = f'{out} {ln}'
    return re.sub(r'\s+', ' ', out).strip()

# Whole lines that are layout noise. Loose patterns because OCR reads the SPD header differently on each page
NOISE = {'*':   [r'\d{1,3}', r'https?://\S+'],          # numbers, links
         'ano': [r'Zadavatel:.*'],
         'spd': [r'.*Program\s*[—–-]+\s*SPD.*', r'sp'],
         # timeline labels glued together, e.g. "DO HODINY DO 24 HODINDO 24 HODIN DO MĚSÍCE... - added after review"
         'spolu-portlik': [r'(DO\s*(HODINY|\d+\s*HODIN|MĚSÍCE|ROKA|ROKU\s*\d{4})\s*)+']}

def clean_lines(party, text):
    keep = []
    for ln in text.split('\n'):
        s = ln.strip()
        if any(re.fullmatch(p, s) for p in NOISE['*'] + NOISE.get(party, [])):
            continue
        keep.append(re.sub(r'^[«+•▪●■]\s*', '- ', s))       # OCR reads bullets as « or +
    return '\n'.join(keep)

JSON inspect.

Because we have Pirates party in the corpus whose file structure is JSON, it is necessary to deal with the different file format.

In [ ]:
# E2a | Print every key path of the pirati JSON with an example value
def walk(x, path=''):
    if isinstance(x, dict):
        for k, v in x.items(): yield from walk(v, f'{path}.{k}')
    elif isinstance(x, list):
        for v in x: yield from walk(v, f'{path}[]')
    else:
        yield path, str(x)

data = json.loads((RAW_DIR / FILES['pirati']).read_text(encoding='utf-8'))
paths = pd.DataFrame(walk(data), columns=['path', 'value'])
print(paths.groupby('path')['value'].agg(n='count', example=lambda s: s.iloc[0][:70]).to_string())

From the output of the previous code block, it is necessary to identify dates and URLs to remove them from corpus.

In [ ]:
# E2b | Readers: each returns a list of blocks (one block ~ one paragraph)
from striprtf.striprtf import rtf_to_text

def blk(party, page, text, source='text_layer', size_ratio=None, level=None):
    return dict(party=party, source_doc=FILES[party], page=page, text=text,
                source=source, size_ratio=size_ratio, level=level)

def read_pdf(party):
    blocks = []
    for pno, page in enumerate(fitz.open(RAW_DIR / FILES[party]), start=1):
        for b in page.get_text('dict')['blocks']:            # PyMuPDF block ~ paragraph (fixes P5 levice)
            if b['type'] != 0: continue                       # 1 = image block
            lines = [''.join(s['text'] for s in l['spans']) for l in b['lines']]
            sizes = [s['size'] for l in b['lines'] for s in l['spans'] if s['text'].strip()]
            text = join_lines(clean_lines(party, '\n'.join(lines)).split('\n'))
            if text and sizes:
                blocks.append(blk(party, pno, text, size_ratio=max(sizes)))
    body = pd.Series([b['size_ratio'] for b in blocks]).round(1).mode()[0]  # most common size = body text
    for b in blocks: b['size_ratio'] = round(b['size_ratio'] / body, 2)
    return blocks

def read_plain(party, text, page=1, source='text_layer'):
    paras = re.split(r'\n\s*\n', clean_lines(party, text))   # paragraph = text between empty lines
    return [blk(party, page, join_lines(p.split('\n')), source) for p in paras if p.strip()]

def read_txt(party):
    raw = '\n'.join(corpus.loc[corpus.party == party, 'text'])   # whole file is one "page 1" record
    if raw.lstrip().startswith('{\\rtf'):                        # voluntia: RTF disguised as .txt (P2)
        raw = rtf_to_text(raw)
    return read_plain(party, raw)

def read_ocr(party):
    rows = corpus[(corpus.party == party) & (corpus.source == 'ocr')]
    return [b for _, r in rows.iterrows() for b in read_plain(party, r['text'], int(r['page']), 'ocr')]

HEAD_KEYS = {'title', 'heading'}     # <- set after E2a
TEXT_KEYS = {'text', 'content'}      # <- set after E2a
def read_json(party):
    data = json.loads((RAW_DIR / FILES[party]).read_text(encoding='utf-8'))
    blocks = []
    def visit(x, depth):
        if isinstance(x, dict):
            for k, v in x.items():
                if isinstance(v, str) and k in HEAD_KEYS:
                    blocks.append(blk(party, 1, v.strip(), level=depth))   # remember depth for now
                elif isinstance(v, str) and k in TEXT_KEYS:
                    blocks.extend(read_plain(party, v))
                else:
                    visit(v, depth + 1)
        elif isinstance(x, list):
            for v in x: visit(v, depth)
    visit(data, 0)
    depths = sorted({b['level'] for b in blocks if b['level'] is not None})
    for b in blocks:                                          # shallowest heading depth -> H1, deeper -> H2
        if b['level'] is not None: b['level'] = 1 if b['level'] == depths[0] else 2
    return blocks

In [ ]:
# E2c | Pirati reader: only programme text, by explicit key path (replaces the generic read_json)
#   areas[].title -> H1, areas[].points[].title -> H2, items[].text -> body (+ item_type)
#   sources[].title, situations[], questions[], people[] are ignored on purpose
def end_dot(t):
    """JSON list items have no final period; add one so merged items stay separate sentences.
    Note: this '.' is ours, not in the source - quotes are checked against passage text."""
    t = t.strip()
    return t if t[-1:] in '.!?…:' else t + '.'

def read_pirati(party='pirati'):
    data = json.loads((RAW_DIR / FILES[party]).read_text(encoding='utf-8'))
    blocks = []
    def item(it):
        b = blk(party, 1, end_dot(join_lines(it['text'].split('\n'))))
        b['item_type'], b['item_id'] = it.get('type'), it.get('id')   # problem / result / ...
        return b
    for area in data['areas']:
        blocks.append(blk(party, 1, area['title'].strip(), level=1))
        blocks += [item(it) for it in area.get('items', [])]           # area-level items
        for pt in area.get('points', []):
            blocks.append(blk(party, 1, pt['title'].strip(), level=2))
            blocks += [item(it) for it in pt.get('items', [])]
    return blocks

pb = read_pirati()
print(len(pb), 'blocks')
print(pd.Series([b.get('item_type') for b in pb]).value_counts(dropna=False))   # None = headings

### Why `item_type`? (pirati only)

The Pirates' JSON labels every programme item with a type. These labels come from the
party itself, not from us. The 636 blocks are:

| type | n | meaning | is it a party position? |
|---|---|---|---|
| `promise` | 387 | what they will do ("Postavíme městské byty…") | yes |
| `position` | 24 | an explicit stance | yes |
| `goal` | 7 | a target they commit to | yes |
| `ongoing` | 15 | work already in progress that they will continue | yes, implicitly |
| `problem` | 65 | a description of the current situation ("Praha má málo obecních bytů…") | **no** |
| `result` | 65 | an achievement from a past term | **no** (a record, not a programme) |
| `None` | 73 | headings (11 areas + 62 points) | — |

**Why we keep it.** The stance step asks whether a party AGREES or DISAGREES with a thesis.
A `problem` or `result` sentence uses the same words as a position, so retrieval will find it.
A model may then read a problem description as a stance. With the label we can:
1. keep these items in the corpus (they add context for clustering and reading),
2. exclude `problem` and `result` from the evidence used for stance,
3. use them as a free error check: the stance model should return NO POSITION when only such items are retrieved.

**Limitation.** Only pirati has these labels. Other parties mix problems and promises in
plain text, so for them the stance model must make this distinction itself. This gap is
reported as an asymmetry in the limits chapter. We do not hand-label the other parties.

### Noise identification
Layout noise is text that comes from the document format, not from the programme. Examples are running headers, page numbers, URLs, map labels and timeline labels.

Slogans, problem descriptions and vague statements are programme content and are never treated as noise. They are handled later: by item_type for pirati, and by the stance step for all parties.

In [ ]:
# E2d | Noise rules per block. A noise block is dropped from passages but kept in all_blocks (trace).
P['NAME_SHARE'] = 0.7     # N2: share of Title-case words/numbers in a "list of names"
P['MIN_WORDS']  = 6       # N3 (passage level): shorter passages are flagged

def norm(t): return re.sub(r'[\d\W_]+', ' ', t.lower()).strip()

def mark_noise(blocks):
    pages = {}
    for b in blocks: pages.setdefault(norm(b['text']), set()).add(b['page'])
    for b in blocks:
        w = b['text'].split()
        b['noise'] = ''
        if b['level'] is not None:        # JSON headings are never noise
            continue
        if len(w) <= 12 and len(pages[norm(b['text'])]) >= 3:
            b['noise'] = 'N1 repeated on >=3 pages'            # running headers
        elif len(w) >= 10 and sum(x.istitle() or x.isdigit() for x in w) / len(w) >= P['NAME_SHARE']:
            b['noise'] = 'N2 list of names'                     # e.g. prahasobe district map
    return blocks

E3 + F: Headings, grouping, splitting

In [ ]:
# ---- V1: raw snapshot of the SPD page = provenance for v4 ----
import requests, hashlib, json, re
SPD_URL  = "https://www.spdpraha.cz/program/"
spd_html = RAW_DIR / "spd_program.html"
if not spd_html.exists():
    r = requests.get(SPD_URL, headers={"User-Agent": "TextAnalyticsCourseProject/1.0"}, timeout=30)
    r.raise_for_status(); spd_html.write_bytes(r.content)
for f in (spd_html, RAW_DIR / "voluntia_sections.json"):       # paste these into sources.csv
    print(f.name, "| sha256:", hashlib.sha256(f.read_bytes()).hexdigest(), "| saved:", f.stat().st_mtime)

In [ ]:
# ---- V2: readers -> blocks (level 1 = title, 2 = sub-title, 0 = body); same keys as blk() ----
from bs4 import BeautifulSoup

def squash(s): return re.sub(r"\s+", " ", s).strip()

LABELS = {"my chceme"}

def hb(party, doc, text, level):
    return dict(party=party, source_doc=doc, page=1, text=text, source="html", size_ratio=None, level=level)

def read_voluntia(party="voluntia"):
    V, blocks = json.loads((RAW_DIR / "voluntia_sections.json").read_text(encoding="utf-8")), []
    for sec in V["sections"]:
        blocks.append(hb(party, "voluntia_sections.json", sec["title"], 1))
        paras = [join_lines(p.split("\n")) for p in re.split(r"\n\s*\n", sec["text"]) if p.strip()]
        for k, t in enumerate(paras):
            nxt = n_words(paras[k + 1]) if k + 1 < len(paras) else 0
            sub = n_words(t) <= 10 and t[-1] not in ".,;:!?" and nxt >= 2 * n_words(t)   # short, no full stop, long text after
            blocks.append(hb(party, "voluntia_sections.json", t, 2 if sub else 0))
    return blocks

def spd_lines(p):
    """One <p> -> [(text, is_bold)]; <br> ends a line, <strong> is always its own line."""
    out, cur = [], ""
    for node in p.children:
        name = getattr(node, "name", None)
        if name == "br" or name == "strong":
            if cur.strip(): out.append((squash(cur), False)); cur = ""
            if name == "strong": out.append((squash(node.get_text(" ")), True))
        else:
            cur += node.get_text(" ") if name else str(node)
    if cur.strip(): out.append((squash(cur), False))
    return out

def read_spd(party="spd"):
    soup = BeautifulSoup(spd_html.read_text(encoding="utf-8"), "html.parser")
    for t in soup(["script", "style", "nav", "footer"]): t.decompose()
    root, blocks = soup.find("main") or soup.body, []
    for p in root.find_all("p", class_="wp-block-paragraph"):
        for t, bold in spd_lines(p):
            lvl = (1 if re.match(r"\d{1,2}\.\s", t) else 2) if bold and t[-1] not in ".:" and t.lower() not in LABELS else 0
            blocks.append(hb(party, "spd_program.html", re.sub(r"^•\s*", "- ", t), lvl))
    first = next(i for i, b in enumerate(blocks) if b["level"] == 1)      # drop the cover slogan
    return blocks[first:]

for name, bl in {"voluntia": read_voluntia(), "spd": read_spd()}.items():
    print(f"\n{name}: {len(bl)} blocks | words: {sum(n_words(b['text']) for b in bl)}")
    for b in bl:
        if b["level"]: print("  " * b["level"], f"H{b['level']}:", b["text"][:70])

In [ ]:
# E3+F | Heading level + rule per block, then passages under the nearest heading
NUM = re.compile(r'^(\d{1,2}(?:\.\d{1,2})*)\.?\s+\S')    # "3 DOPRAVA", "3. BYDLENÍ", "1.1 Více ..."
SENT = re.compile(rf'(?<=[.!?])\s+(?=[{UP}])')            # sentence end followed by a capital letter
P['NUM_HEAD_MAX_WORDS'] = 25

def heading_level(b, nxt):
    """Returns (level, rule). level: 0 body, 1 top heading, 2 sub-heading. First matching rule wins."""
    t, w = b['text'], n_words(b['text'])
    if b['level'] is not None:                       return b['level'], 'R1 json structure'
    if t.startswith('#'):                            return min(len(t) - len(t.lstrip('#')), 2), 'R2 markdown #'
    m = NUM.match(t)
    if m and w <= P['NUM_HEAD_MAX_WORDS'] and t[-1] not in '.,;:':
        return (1 if '.' not in m.group(1) else 2), f'R3 numbering "{m.group(1)}"'
    if w > P['HEAD_MAX_WORDS']:                      return 0, f'R4 body: {w} words > {P["HEAD_MAX_WORDS"]}'
    if t.startswith('- '):                           return 0, 'R5 body: bullet'
    if t[-1] in '.,;:':                              return 0, f'R6 body: ends with "{t[-1]}"'
    if b['size_ratio'] is not None:                  # PDF only: font size relative to body text
        r = b['size_ratio']
        if r >= P['H1_RATIO']: return 1, f'R7 font {r} >= {P["H1_RATIO"]}'
        if r >= P['H2_RATIO']: return 2, f'R7 font {r} >= {P["H2_RATIO"]}'
        return 0, f'R8 body: font {r} < {P["H2_RATIO"]}'
    if nxt is not None and n_words(nxt['text']) >= 2 * w:
        return (1 if t.isupper() else 2), 'R9 short before long'
    return 0, 'R10 body: default'

def split_long(text):
    """Split a paragraph longer than MAX_WORDS at sentence ends."""
    if n_words(text) <= P['MAX_WORDS']: return [text]
    parts, cur = [], ''
    for s in SENT.split(text):
        if cur and n_words(cur) + n_words(s) > P['MAX_WORDS']:
            parts.append(cur); cur = s
        else:
            cur = f'{cur} {s}'.strip()
    return parts + [cur]

def build_passages(blocks):
    rows, h1, h2, hb1, hb2 = [], '', '', None, None
    for i, b in enumerate(blocks):
        if b.get('noise'):                                          # skip noise blocks
            b['is_heading'], b['rule'] = False, b['noise']
            continue
        lvl, rule = heading_level(b, blocks[i + 1] if i + 1 < len(blocks) else None)
        b['is_heading'], b['rule'] = lvl > 0, rule
        t = b['text'].lstrip('# ')
        if lvl == 1:   h1, h2, hb1, hb2 = t, '', b['bid'], None
        elif lvl == 2: h2, hb2 = t, b['bid']
        else:
            for s in split_long(b['text']):
                rows.append({**b, 'heading_1': h1, 'headline': h2 or h1, 'text': s,
                             'bids': [b['bid']], 'headline_bid': hb2 or hb1})
    out = []
    for r in rows:
        key = lambda x: (x['heading_1'], x['headline'], x.get('item_type'))   # item_type in key
        same = out and key(r) == key(out[-1])
        if same and n_words(out[-1]['text']) + n_words(r['text']) <= P['MAX_WORDS']:
            out[-1]['text'] += ' ' + r['text']; out[-1]['bids'] += r['bids']
        else:
            out.append(dict(r))
    return out

# ---- which reader reads which source ----
READERS  = {'pdf': read_pdf, 'txt': read_txt}                                   # by extension
BY_PARTY = {'pirati': read_pirati, 'voluntia': read_voluntia, 'spd': read_spd}  # sources with their own structure

def read_any(party, fname):
    if party in OCR_PARTIES: return read_ocr(party)
    return (BY_PARTY.get(party) or READERS[fname.rsplit('.', 1)[1]])(party)

no_reader = [(p, f) for p, f in FILES.items()
             if p not in OCR_PARTIES and p not in BY_PARTY and f.rsplit('.', 1)[1] not in READERS]
assert not no_reader, (f"No reader for {no_reader}. If a party name looks wrong (e.g. voluntia_sections), "
                       "the raw corpus is out of date: rerun phase 1 (cells 5-19) and E0 after renaming files.")

all_rows, all_blocks = [], []
for party, fname in FILES.items():
    blocks = read_any(party, fname)
    for i, b in enumerate(blocks): b['bid'] = f'{party}_{i:04d}'
    blocks = mark_noise(blocks)
    all_rows += build_passages(blocks); all_blocks += blocks

df = pd.DataFrame(all_rows)
df['number_of_words'] = df['text'].map(n_words)
df['idx'] = df.groupby(['party', 'page']).cumcount() + 1
df['passage_id'] = [f'{p}_{pg:03d}_{i:03d}' for p, pg, i in zip(df.party, df.page, df.idx)]
df['sha256'] = df['party'].map(SHA)
df['is_noise'] = df['number_of_words'] < P['MIN_WORDS']                     # N3, flagged not dropped
print(len(df), 'passages;', len(all_blocks), 'blocks;', df.is_noise.sum(), 'short passages flagged')

noise count and review

In [ ]:
# F1 | How much did each noise rule remove? Review the texts before accepting the rules.
nb = pd.DataFrame(all_blocks)
nb['words'] = nb['text'].map(n_words)
print(nb.assign(noise=nb.noise.replace('', 'kept'))
        .pivot_table(index='noise', columns='party', values='words', aggfunc='sum', fill_value=0))
for rule, g in nb[nb.noise != ''].groupby('noise'):
    print(f'\n{rule}:'); [print('  -', t[:90]) for t in g.text.drop_duplicates().head(8)]

def show(label, pattern, party=None):
    s = df if party is None else df[df.party == party]
    hit = s[s.text.str.contains(pattern, regex=False)]
    if hit.empty: return print(f'{label}: not found')
    t = hit.text.iloc[0]; i = t.find(pattern)
    print(f'{label}:  ...{t[max(0, i - 40):i + 60]}...')

show('CHECK pirati join  (expect "Řep. V")', 'Řep')
show('CHECK levice hyphen (bug while "středněa" appears)', 'středn', 'levice')

# Corpus composition

First we create the corpus as it is collected from the raw programmes. Then as we find mistakes we fix them in this part of the notebook and update the corpus version.

We chose this approach as it is the most transparent and in our opinion also the most synoptical approach for reviews to follow along our processes.

This cell is necessary for the scraping and automatic saving of HTML files.

In [ ]:
# F2 | Freeze the passage table. A frozen version is NEVER overwritten.
#      topic_general is NOT stored here: notebook 02 (block T0) labels every row of every corpus version.
#      To change rules later: set VERSION='v2' and use text_hash to map old IDs to new ones.
import hashlib
VERSION = 'v1'
out = CORPUS_DIR / f'corpus_{VERSION}.csv'
df['text_hash'] = df.text.map(lambda t: hashlib.sha1(t.encode('utf-8')).hexdigest()[:12])
cols = ['passage_id', 'party', 'source_doc', 'page', 'source', 'heading_1', 'headline', 'text',
        'number_of_words', 'item_type', 'is_noise', 'text_hash', 'sha256']

if out.exists():
    old = pd.read_csv(out)
    same = set(old.text_hash) & set(df.text_hash)
    print(f'{out.name} is frozen - not overwritten. {len(same)}/{len(old)} passages unchanged in this run.')
else:
    assert df.passage_id.is_unique
    df[cols].to_csv(out, index=False)
    print(f'saved {len(df)} passages -> {out.name}')


out = CORPUS_DIR / "corpus_v0_raw.jsonl"
key = lambda d: set(zip(d["doc_id"], d["sha256"], d["n_words"]))
if out.exists():   # frozen: never overwrite; later cells read this file, so it must match this run
    assert key(pd.read_json(out, lines=True)) == key(corpus), f"{out.name} differs from this run. Rename it, then rerun."
    print(f"{out.name} exists and matches ({len(corpus)} records). Not overwritten.")
else:
    corpus.to_json(out, orient="records", lines=True, force_ascii=False)   # keeps Czech diacritics
    print(f"saved {len(corpus)} records -> {out.name}")


## Notes

### What we call noise and how we handle it
Layout noise is text produced by the document format rather than by the party’s programme: running headers, page numbers, URLs, map labels and timeline labels. It is removed by four transparent rules (N1, N2 and two per-party patterns); each removed block keeps its rule and the removed word counts are reported per party. Passages shorter than 6 words are flagged but kept. Slogans, problem descriptions and vague statements are programme content and are never treated as noise. Passages that HDBSCAN leaves outside every cluster (label −1) are called unclustered; they remain available for retrieval.

### Frozen corpus
After each major change of corpus, we create a new version. We do not rewrite the same corpus file all over again.


**Why these choices**

- One P dict: a teammate who re-runs the notebook gets the same numbers, and the report can quote one table. Rejected: literals scattered across cells, which drift apart over time.
- `is_noise` normalisation: a CSV round trip can make the boolean column text, and then `~df["is_noise"]` gives the wrong rows without any error. This is a real trap, so the check is worth the line.
- Starting UMAP and HDBSCAN values: they are not tuned yet. min_dist=0.0 is the usual choice for clustering, because UMAP’s default of 0.1 is meant for plotting. Tuning happens only in C3, on cluster metrics, and every change is logged.
- Rejected: a separate noise-free corpus file. It would duplicate the frozen data and create a second source of truth.

## Praha Sobě failiure
In the first version, Praha Sobě was causing problems with classification as their programme is written in three columns and the PDF reader was reading it wrong. It was flagged after the embedding results came out and was solved by editing and pasting the programme in text by hand.

Martin Snizek is responsible for the way the programme was edited.


In [ ]:
# One shared PASS/FAIL helper. FAILS collects problems so a cell can stop the notebook.
FAILS = []
def report(ok, msg):
    print(("PASS  " if ok else "FAIL  ") + msg)
    if not ok: FAILS.append(msg)

def l2norm(M):
    return M / np.linalg.norm(M, axis=1, keepdims=True)

def to_bool(s):
    # CSV round trip can turn booleans into text; this avoids a silently wrong noise filter
    return s.astype(str).str.lower().isin(["true", "1"])

**Checker of the edited file**

**Purpose:** Reads the hand-edited Praha sobě file and checks its structure. It only reads and prints. It never changes or saves anything.

### What the file must look like
- Plain text (`.txt` or `.md`).
- One programme item = one block of lines.
- Blocks are separated by an **empty line**.
- The first line of a block is the heading: a number and a dot, e.g. `18. MĚSTSKÉ POZEMKY ...`.
- The file has two parts (A and B). Numbering restarts at 1 in each. The detail part is the one that reaches item 57.

## How it works
1. Finds the file under `PROJECT_DIR` and cleans tabs and non-breaking spaces.
2. Cuts the text into blocks at empty lines.
3. Reads the heading number of each block.
4. Starts a new part whenever the number does not go up (e.g. 18 → 1).
5. Runs the checks below and prints `PASS` or `FAIL`.


In [ ]:
# ---- 2. X3: checks for prahasobe_manual (read only) ----
UP = "A-ZÁČĎÉĚÍŇÓŘŠŤÚŮÝŽ"
FAILS.clear()

hits = glob.glob(os.path.join(PROJECT_DIR, "**", "prahasobe_manual*"), recursive=True)
assert len(hits) == 1, f"Expected exactly one prahasobe_manual file, found: {hits}"
MANUAL_PATH = hits[0]
print("File:", MANUAL_PATH)

raw = open(MANUAL_PATH, encoding="utf-8-sig").read().replace("\xa0", " ").replace("\t", " ")
blocks = [b for b in re.split(r"\n\s*\n", raw) if b.strip()]    # one block = one item

HEAD = re.compile(r"^\s*(\d{1,2})\.\s*\S")
HEAD_CAPS = re.compile(rf"^\s*\d{{1,2}}\.\s*[{UP}]{{2,}}")

rows, part, prev = [], 0, 0
for b in blocks:
    lines = [l.strip() for l in b.split("\n") if l.strip()]
    m = HEAD.match(lines[0])
    n = int(m.group(1)) if m else None
    if n is not None:
        if n <= prev: part += 1           # numbering restarted -> next part
        prev = n
    rows.append({"part": part if n is not None else None, "item_no": n,
                 "heading": lines[0], "body": " ".join(lines[1:]),
                 "inner_heading": any(HEAD_CAPS.match(l) for l in lines[1:])})
df_manual = pd.DataFrame(rows)
df_manual["words"] = df_manual["body"].str.split().str.len()
numbered = df_manual.dropna(subset=["item_no"]).copy()
numbered["item_no"] = numbered["item_no"].astype(int)

print("\nParts found:")
for p, g in numbered.groupby("part"):
    print(f"  part {int(p)}: {len(g)} items, numbers {g['item_no'].min()}-{g['item_no'].max()}")
for p, g in numbered.groupby("part"):
    nums = g["item_no"].tolist()
    dups = sorted({x for x in nums if nums.count(x) > 1})
    gaps = sorted(set(range(1, max(nums) + 1)) - set(nums))
    report(not dups and not gaps, f"Part {int(p)}: no duplicates {dups}, no gaps {gaps}")

detail = numbered[numbered["part"] == numbered.loc[numbered["item_no"].idxmax(), "part"]]
report(detail["item_no"].max() == 57 and len(detail) == 57, f"Detail part has 57 items (has {len(detail)})")
report((numbered["words"] > 0).all(), "Every numbered item has a body")
report(not df_manual["inner_heading"].any(),
       "No second heading inside a block: " + str(df_manual.loc[df_manual["inner_heading"], "item_no"].tolist()))
report(df_manual["item_no"].notna().all(), "Every block has a numbered heading")
alltext = " ".join(df_manual["heading"] + " " + df_manual["body"])
report("facebook.com" not in alltext and "instagram.com" not in alltext, "No footer links")
report("Praha 19 Praha 20" not in alltext, "No district-name block from the map")


assert not FAILS, f"prahasobe_manual has {len(FAILS)} problem(s). Fix the file and rerun: {FAILS}"

### Corpus V2 composition

In [ ]:
# ---- X5: corpus_v2 = frozen corpus_v1 without old prahasobe rows + items from prahasobe_manual (X3) ----

PS = ["prahasobe", "prahasobe_manual"]       # drop old Praha sobě rows whatever their label
#keep = v1[~v1["party"].isin(PS)]
#meta = v1[v1["party"].isin(PS)].iloc[0]

FAILS.clear()
V1_PATH, V2_PATH = (os.path.join(CORPUS_DIR, f"corpus_{v}.csv") for v in ("v1", "v2"))
def squash(s): return re.sub(r"\s+", " ", s).strip()

PS = "prahasobe_manual"        # <- set to the exact label printed above (e.g. "praha_sobe")
v1 = pd.read_csv(V1_PATH, dtype={"passage_id": str, "text_hash": str})
assert (v1["party"] == PS).any(), f"No '{PS}' in v1. Labels: {sorted(v1['party'].unique())}"
keep = v1[v1["party"] != PS]
meta = v1[v1["party"] == PS].iloc[0]

new = numbered.copy()                                    # one row per manual item, from X3
new["part"] = np.where(new["part"] == detail["part"].iloc[0], "detail", "roadmap")
new["headline"] = [squash(re.sub(r"^\s*\d{1,2}\.\s*", "", h)) for h in new["heading"]]   # title without its number
new["text"] = [squash(f"{h} {b}") for h, b in zip(new["headline"], new["body"])]
new = new.assign(
    passage_id=[f"prahasobe_{p}_{n:03d}" for p, n in zip(new["part"], new["item_no"])],
    party="prahasobe", source_doc=meta["source_doc"], source=meta["source"], sha256=meta["sha256"],
    heading_1=new["part"],                               # roadmap / detail
    number_of_words=new["text"].str.split().str.len(),
    text_hash=[hashlib.sha1(t.encode("utf-8")).hexdigest()[:12] for t in new["text"]])
new["is_noise"] = new["number_of_words"] < 6
df_v2 = pd.concat([keep, new.reindex(columns=v1.columns)], ignore_index=True)   # missing columns (page, item_type) -> NaN

report(df_v2["passage_id"].is_unique, "passage_id is unique")
report(df_v2["text_hash"].is_unique, "text_hash is unique")
print(f"Total {len(df_v2)} = other parties {len(keep)} + prahasobe {len(new)} {new['part'].value_counts().to_dict()}")
assert not FAILS, f"Build checks failed: {FAILS}"

# Frozen: never overwrite an existing v2
key = lambda d: set(zip(d["passage_id"], d["text_hash"]))
if os.path.exists(V2_PATH):
    assert key(pd.read_csv(V2_PATH, dtype=str)) == key(df_v2), "corpus_v2.csv differs. Do not overwrite: rename it, then rerun."
    print("corpus_v2.csv exists and matches. Not overwritten.")
else:
    df_v2.to_csv(V2_PATH, index=False); print("Saved:", V2_PATH)

## Starostove failiure
The problem is with the structure of the programme. The text is structured in blocks and the order in which computer reads the pdf does not correspond to the order human eye reads the text. It is visible in output of the cell below.

Computer sees the rows in different order than human eye, given the programme has 29 pages and the fix is not easy we used data about position of the text on each page to reconstruct the correct order of text for each page.

This was necessary to be done if we wanted to split the text into programme claims correctly.

In [ ]:
# ---- X6: inspect starostove raw pages (read only) ----
RAW_JSONL = os.path.join(CORPUS_DIR, "corpus_v0_raw.jsonl")
raw = pd.read_json(RAW_JSONL, lines=True)
st = raw[raw["party"] == "starostove"].sort_values("page").reset_index(drop=True)
print("pages:", len(st), "| words:", int(st["n_words"].sum()))

def lines_of(t): return [l.strip() for l in t.replace("\xa0", " ").split("\n") if l.strip()]

for _, r in st.iterrows():
    L0 = lines_of(r["text"])
    n_num = sum(bool(re.match(r"^\d+\s*/", l)) for l in L0)
    print(f"p{int(r['page']):>2} | lines {len(L0):>3} | priorities {any('Naše priority' in l for l in L0)} "
          f"| co uděláme {sum('Co uděláme' in l for l in L0)} | N/ lines {n_num} | {L0[0][:35]!r}")

i = st.index[st["text"].str.contains("Jasná pravidla", regex=False)][0]
print("\nFirst 30 lines of the Sport page:")
for l in lines_of(st.at[i, "text"])[:30]: print(repr(l))

The problem is that the lines how the tool for PDF extraction read them, are in different order than how human eyes reads them. It is therefore difficult to extract headings and corresponding text with them.

We solved that by extracting position of the text and font size from the page. With this information we reconstruct the page.

In [ ]:
# ---- X6b: re-read starostove PDF, lines sorted by position on the page ----
import fitz
def squash(s): return re.sub(r"\s+", " ", s).strip()

hits = glob.glob(os.path.join(PROJECT_DIR, "**", "starostove_program.pdf"), recursive=True)
assert len(hits) == 1, f"Expected one starostove_program.pdf, found: {hits}"
doc = fitz.open(hits[0])

PL = []                                   # (page, text, font size) in visual order
for pno, page in enumerate(doc, start=1):
    items = []
    for b in page.get_text("dict")["blocks"]:
        for ln in b.get("lines", []):
            t = squash("".join(s["text"] for s in ln["spans"]))
            if t:
                items.append((round(ln["bbox"][1] / 4), ln["bbox"][0], t, max(s["size"] for s in ln["spans"])))
    PL += [(pno, t, sz) for _, _, t, sz in sorted(items, key=lambda x: (x[0], x[1]))]

print("pages:", len(doc), "| lines:", len(PL))
first = next(p for p, t, _ in PL if "Jasná pravidla" in t)
print(f"\nPage {first} in visual order:")
for p, t, sz in PL:
    if p == first: print(f"{sz:5.1f} | {t[:95]}")

*Split the Starostové programme into passages*

**What it does**
Cuts the Starostové PDF into one passage per claim, using the font size of each line.
It uses the lines from X6b, which are sorted by their position on the page.
Pages 1-4 (cover, letter, overview) are not used. The overview repeats the priorities
that we already have on the topic pages.

**Rules (font size in pt)**
| Size | What it is | What we do |
|---|---|---|
| 30-40 | Topic title, e.g. "Sport" | Starts a new topic |
| 17.5-19.5, no number | One-sentence topic claim | Passage `topic_intro` |
| 12-17.5 after "Naše priority" | Priorities box | One passage per topic, `priority` |
| 17.5-19.5 with `1/` | Claim heading | Starts a passage `claim` |
| below 12 | Bullets | Body of the open claim |

The label "Co uděláme", bullet signs, page headers and page numbers are removed.
All other words stay exactly as in the PDF (also the ironic ones).

**Output**
Table `sd`, one row per passage: `kind`, `topic`, `page`, `num`, `headline`, `text`, `words`.
X8 turns it into corpus_v3.

**Checks**
- Every line is copied word for word from the PDF.
- No label or page header is left.
- Every claim has text.
- Claims are numbered 1, 2, 3... in every topic.

**Known limits**
- The font sizes come from this one PDF. Another PDF needs new ranges.
- Priorities repeat the claims in shorter form. Filter `kind == "priority"` if we want no duplicates.
- Zdravotnictví has one detailed claim, so the new hospital in Letňany is only in the priorities passage.
- Rejected: matching by order in the text layer (wrong order in the PDF), an LLM (not repeatable), manual editing (29 pages).
---



In [ ]:
# ---- X7: split starostove into passages by font size ----
FAILS.clear()
FIRST_PAGE = 5                                           # pages 1-4 (cover, letter, overview) are not used
TITLE, HEAD, BOX, BODY = (30, 40), (17.5, 19.5), (12, 17.5), 12     # font size ranges in pt

ST_NUM   = re.compile(r"^(\d{1,2})\s*/\s*(.+)$")
ST_NOISE = re.compile(r"(STAROSTOVÉ PRO PRAHU|PROGRAM 2026|\d{1,3}|\s)+", re.I)
def clean(t): return re.sub(r"^Co uděláme\s*:?\s*", "", re.sub(r"^[•\u2022\u25cf\uf0b7\-–—]\s*", "", t)).strip()

lines = [(p, clean(t), s) for p, t, s in PL if p >= FIRST_PAGE]
lines = [(p, t, s) for p, t, s in lines if t and not ST_NOISE.fullmatch(t)]

rows, topic, state, cur, intro, tpage = [], None, None, None, [], None
def add(kind, num, head, body, page):
    r = dict(kind=kind, topic=topic, page=page, num=num, headline=head, body=body); rows.append(r); return r

i = 0
while i < len(lines):
    p, t, s = lines[i]; i += 1
    if re.fullmatch(r"\d{1,2}\s*/", t) and i < len(lines):          # lone "1/" -> join with the next line
        t, s = t + " " + lines[i][1], lines[i][2]; i += 1
    m, is_head = ST_NUM.match(t), HEAD[0] <= s <= HEAD[1]

    if TITLE[0] <= s <= TITLE[1]:                                   # big topic title (may have two lines)
        topic = (topic + " " + t) if state == "title" else t
        topic = topic.replace("Náš program pro Prahu v detailu", "").strip()
        state, cur, intro, tpage = "title", None, [], p
    elif topic is None:
        continue
    elif t.startswith("Naše priority"):                              # priorities box: one passage per topic
        if intro: add("topic_intro", 0, topic, intro, tpage)
        state, cur = "box", add("priority", 0, "Naše priority", [], p)
    elif t.startswith("Garanti"):
        state, cur = "claim", None
    elif state == "box" and BOX[0] < s < BOX[1]:
        cur["body"].append(t)
    elif m and is_head:                                              # claim heading "1/ Title"
        state, cur = "claim", add("claim", int(m.group(1)), m.group(2).strip(), [], p)
    elif is_head and state in ("title", "intro"):                    # one-sentence topic claim
        state = "intro"; intro.append(t)
    elif is_head and cur and cur["kind"] == "claim" and not cur["body"]:
        cur["headline"] += " " + t                                   # heading wrapped over two lines
    elif cur and s < BODY:                                           # bullets belong to the open claim
        cur["body"].append(t)

sd = pd.DataFrame(rows)
sd["text"] = [squash(" ".join(([r["headline"]] if r["kind"] == "claim" else []) + r["body"])) for r in rows]
sd["words"] = sd["text"].str.split().str.len()

# ---- checks ----
raw_all = squash(" ".join(t for _, t, _ in PL))
cl = sd[sd["kind"] == "claim"]
report(all(squash(x) in raw_all for r in rows for x in r["body"]), "every body line is verbatim in the PDF text")
report(not any(b in " ".join(sd["text"]) for b in ["Co uděláme", "Garanti programu", "STAROSTOVÉ PRO PRAHU", "PROGRAM 2026"]),
       "no label or page header left")
report((cl["words"] > cl["headline"].str.split().str.len()).all(), "every claim has body text")
bad = [t for t, g in cl.groupby("topic") if g["num"].tolist() != list(range(1, len(g) + 1))]
report(not bad, f"claims numbered 1..n in every topic (problems: {bad})")

raw_words = sum(len(t.split()) for p, t, _ in PL if p >= FIRST_PAGE)
print(f"\nPassages: {len(sd)} | claims: {len(cl)} | topics: {sd['topic'].nunique()}")
print(f"Words kept: {int(sd['words'].sum())} of {raw_words} on pages >= {FIRST_PAGE} ({sd['words'].sum() / raw_words:.0%})")
print(sd.pivot_table(index="topic", columns="kind", values="words", aggfunc=["count", "sum"], fill_value=0, sort=False).to_string())
print("Longest claims (words):", cl["words"].nlargest(3).tolist())

In [ ]:
# ---- D1: where did the missing words go? (approximate match) ----
# A line counts as "kept" when it appears inside a passage text or is a topic title.
kept_text = squash(" ".join(sd["text"]) + " " + " ".join(sd["topic"].dropna().unique()))
all_lines = pd.DataFrame([(p, squash(clean(t))) for p, t, _ in PL], columns=["page", "line"])
all_lines["words"] = all_lines["line"].str.split().str.len()
drop = all_lines[(all_lines["words"] >= 3) & ~all_lines["line"].map(lambda l: l in kept_text)]
print("Words not kept, per page:"); print(drop.groupby("page")["words"].sum().to_string())

n_labels = pd.Series([p for p, t, _ in PL if t.startswith("Co uděláme")]).value_counts().sort_index()
cmp = pd.concat([n_labels.rename("labels"), sd[sd["kind"] == "claim"]["page"].value_counts().sort_index().rename("claims")],
                axis=1).fillna(0).astype(int)
print("\nPages where labels != claims:"); print(cmp[cmp["labels"] != cmp["claims"]].to_string())

This output shows us, where did the algorithm drop words that are not stored into corpus.

`Pages where labels != claims:
    labels  claims`

This output means that there is a different number of labels and claims on these pages. We detect labels by "Co uděláme" text. On page 16 (when reviewing the pdf it is numbered as page 15), the claim overflows to the next page and there is heading but - claim, but the text is missing. On the other hand the next page has 3 labels but only two numbered headings (claims).

### Corpus V3 composition

* Rebuilt parties: ANO, SPD and Voluntia are rebuilt, and spojena_levice and jsme_praha are added.
* Checks: there are no asserts, so nothing stops the run. Checks print PASS or FAIL, and the corpus is always written.
* Failures: a party whose reader fails is skipped with a FAIL line, and its old v3 rows stay in v4.
* Frozen rule: corpus_v4.csv is never overwritten. If it exists and differs,   warning is returned, not an error.



In [ ]:
# ---- X8: build corpus_v3 = corpus_v2 without old starostove rows + new starostove rows ----
FAILS.clear()
V2_PATH = os.path.join(CORPUS_DIR, "corpus_v2.csv"); V3_PATH = os.path.join(CORPUS_DIR, "corpus_v3.csv")
v2 = pd.read_csv(V2_PATH, dtype={"passage_id": str, "text_hash": str}); v2["is_noise"] = to_bool(v2["is_noise"])
old = v2[v2["party"] == "starostove"]; keep = v2[v2["party"] != "starostove"]; meta = old.iloc[0]

sd["idx"] = sd.groupby("page").cumcount() + 1
new = pd.DataFrame({
    "passage_id": [f"starostove_{p:03d}_{i:03d}" for p, i in zip(sd["page"], sd["idx"])],
    "party": "starostove", "source_doc": meta["source_doc"], "source": meta["source"], "sha256": meta["sha256"],
    "page": sd["page"], "heading_1": sd["topic"], "headline": sd["headline"], "text": sd["text"],
    "number_of_words": sd["words"], "item_type": sd["kind"], "is_noise": sd["words"] < 6, "topic_general": np.nan,
    "text_hash": [hashlib.sha1(t.encode("utf-8")).hexdigest()[:12] for t in sd["text"]]})[v2.columns]
df_v3 = pd.concat([keep, new], ignore_index=True)

report(df_v3["passage_id"].is_unique, "passage_id is unique")
report(df_v3["text_hash"].is_unique, "text_hash is unique")
report(len(df_v3) == len(keep) + len(new), f"total {len(df_v3)} = other parties {len(keep)} + starostove {len(new)} (was {len(old)})")
report((df_v3["party"] == "prahasobe").sum() == (v2["party"] == "prahasobe").sum(), "prahasobe rows unchanged")
assert not FAILS, f"Build checks failed: {FAILS}"

if os.path.exists(V3_PATH):
    cur3 = pd.read_csv(V3_PATH, dtype={"passage_id": str, "text_hash": str})
    assert set(zip(cur3["passage_id"], cur3["text_hash"])) == set(zip(df_v3["passage_id"], df_v3["text_hash"])), \
        "corpus_v3.csv exists and differs. Do not overwrite: rename it, then rerun."
    print("corpus_v3.csv already exists and matches. Not overwritten.")
else:
    df_v3.to_csv(V3_PATH, index=False); print("Saved:", V3_PATH)

## ANO, SPD, Voluntia

Patching and scraping the sites, as the pdf reading of SPD via OCR is not really reliable and collects noise.

Voluntia is text and headlines could not be extracted.

ANO is only repaired, as there are headers of the PDF page that are taken as headings of topics, that is fixed here.

In [ ]:
# ---- V3: passage builder for the new blocks + ANO rebuilt from the same PDF ----
NOISE["ano"] = list(dict.fromkeys(NOISE["ano"] + [
    r"PRAHA BUDE ZASE MAKAT",
    r"Volební program ANO 2011 pro ZHMP 2026\s*[-–]\s*zkrácená verze"]))     # running header, pages 2-12

def to_df(blocks, party, sha):
    for i, b in enumerate(blocks): b["bid"] = f"{party}_{i:04d}"
    d = pd.DataFrame(build_passages(blocks))
    d["number_of_words"] = d["text"].map(n_words)
    d["idx"] = d.groupby("page").cumcount() + 1
    d["passage_id"] = [f"{party}_{pg:03d}_{i:03d}" for pg, i in zip(d["page"], d["idx"])]
    d["party"], d["sha256"], d["item_type"] = party, sha, None
    d["is_noise"] = d["number_of_words"] < P["MIN_WORDS"]
    d["text_hash"] = d["text"].map(lambda t: hashlib.sha1(t.encode("utf-8")).hexdigest()[:12])
    return d

sha_file = lambda f: hashlib.sha256(f.read_bytes()).hexdigest()
ano_df = to_df(mark_noise(read_pdf("ano")), "ano", SHA["ano"])
spd_df = to_df(mark_noise(read_spd()), "spd", sha_file(spd_html))
vol_df = to_df(mark_noise(read_voluntia()), "voluntia", sha_file(RAW_DIR / "voluntia_sections.json"))

v3 = pd.read_csv(os.path.join(CORPUS_DIR, "corpus_v3.csv"), dtype={"passage_id": str, "text_hash": str})
old = v3[v3["party"] == "ano"]
print(f"ANO: {len(old)} -> {len(ano_df)} passages | old hashes gone: {len(set(old.text_hash) - set(ano_df.text_hash))}"
      f" | new hashes: {len(set(ano_df.text_hash) - set(old.text_hash))}")
print("ANO header left in text:", int(ano_df["text"].str.contains("MAKAT|zkrácená verze").sum()), "(expect 0)")

### Corpus V4 compilation

In [ ]:
# ---- X9: build corpus_v4 = corpus_v3 without ano/spd/voluntia + rebuilt rows + new parties (spojena_levice, jsme_praha) ----
FAILS.clear()
V3_PATH, V4_PATH = (os.path.join(CORPUS_DIR, f"corpus_{v}.csv") for v in ("v3", "v4"))
v3 = pd.read_csv(V3_PATH, dtype={"passage_id": str, "text_hash": str}); v3["is_noise"] = to_bool(v3["is_noise"])

# ANO running header ("PRAHA BUDE ZASE MAKAT / Volební program ANO 2011 ... zkrácená verze") removed at line level
NOISE["ano"] = list(dict.fromkeys(NOISE["ano"] + [
    r"PRAHA BUDE ZASE MAKAT",
    r"Volební program ANO 2011 pro ZHMP 2026\s*[-–]\s*zkrácená verze"]))

NEW     = ["spojena_levice", "jsme_praha"]                 # never in v1-v3
REBUILT = ["ano", "spd", "voluntia"] + NEW

def to_df(blocks, party):
    """blocks -> passage table with the same columns as the frozen corpus."""
    for i, b in enumerate(blocks): b["bid"] = f"{party}_{i:04d}"
    d = pd.DataFrame(build_passages(blocks))
    d["number_of_words"] = d["text"].map(n_words)
    d["idx"] = d.groupby("page").cumcount() + 1
    d["passage_id"] = [f"{party}_{pg:03d}_{i:03d}" for pg, i in zip(d["page"], d["idx"])]
    d["party"], d["sha256"], d["item_type"] = party, SHA[party], None
    d["is_noise"] = d["number_of_words"] < P["MIN_WORDS"]
    d["text_hash"] = d["text"].map(lambda t: hashlib.sha1(t.encode("utf-8")).hexdigest()[:12])
    return d

dfs = {}
for p in REBUILT:
    if p not in FILES:
        report(False, f"{p}: not in FILES (file missing in Programy/ or phase 1 not rerun)"); continue
    try:
        dfs[p] = to_df(mark_noise(read_any(p, FILES[p])), p)
        report(len(dfs[p]) > 0, f"{p}: {len(dfs[p])} passages from {FILES[p]}")
    except Exception as e:
        report(False, f"{p}: reading failed: {type(e).__name__}: {e}")

keep = v3[~v3["party"].isin(list(dfs))]                    # parties that failed keep their v3 rows
new  = pd.concat(dfs.values(), ignore_index=True).reindex(columns=v3.columns)
df_v4 = pd.concat([keep, new], ignore_index=True)

# ---- checks (print only, never stop) ----
report(df_v4["passage_id"].is_unique, "passage_id is unique")
report(df_v4["text_hash"].is_unique, "text_hash is unique")
report(set(NEW).isdisjoint(v3["party"]), "new parties were not in v3")
report(set(zip(keep.passage_id, keep.text_hash)) == set(zip(v3[v3.party.isin(keep.party.unique())].passage_id,
                                                           v3[v3.party.isin(keep.party.unique())].text_hash)),
       f"other parties unchanged ({len(keep)} rows)")
if "ano" in dfs:
    o = v3[v3.party == "ano"]
    print(f"   ANO: {len(o)} -> {len(dfs['ano'])} passages | old hashes gone: {len(set(o.text_hash) - set(dfs['ano'].text_hash))}"
          f" | new hashes: {len(set(dfs['ano'].text_hash) - set(o.text_hash))}")
    report(not dfs["ano"]["text"].str.contains("MAKAT|zkrácená verze").any(), "ANO running header gone from text")
if "spd" in dfs:
    t = " ".join(dfs["spd"]["text"]) + " " + " ".join(dfs["spd"]["headline"].fillna(""))
    report(not any(j in t for j in ["wove", "spdpraha.cz", "Program — SPD"]), "SPD junk strings gone")
for p in ["spd", "voluntia"] + NEW:
    if p in dfs:
        report(not (dfs[p]["headline"].isna() | dfs[p]["headline"].eq("")).any(), f"{p}: no empty headline")

# ---- comparison v3 vs v4 per party ----
def stats(d):
    g = d.groupby("party")
    return pd.DataFrame({"passages": g.size(), "words": g["number_of_words"].sum(),
                         "empty_headline": g["headline"].apply(lambda s: int((s.isna() | s.eq("")).sum())),
                         "noise_flagged": g["is_noise"].sum()})
cmp = pd.concat({"v3": stats(v3), "v4": stats(df_v4)}, axis=1).fillna(0).astype(int)
cmp.columns = [f"{a}_{b}" for a, b in cmp.columns]
cmp["status"] = ["new" if p in NEW else "rebuilt" if p in dfs else "same" for p in cmp.index]
display(cmp)
for p in NEW:                                              # eyeball heading quality of the new parties
    if p in dfs: print(f"\n{p} headlines:", dfs[p]["headline"].drop_duplicates().head(8).tolist())

# ---- save (a frozen version is never overwritten; no stop on mismatch) ----
print(f"\n{len(FAILS)} check(s) failed: {FAILS}" if FAILS else "\nAll checks passed.")
if os.path.exists(V4_PATH):
    cur = pd.read_csv(V4_PATH, dtype={"passage_id": str, "text_hash": str})
    same = set(zip(cur.passage_id, cur.text_hash)) == set(zip(df_v4.passage_id, df_v4.text_hash))
    print("corpus_v4.csv already exists and matches. Not overwritten." if same else
          "WARNING: corpus_v4.csv exists and differs. NOT overwritten. Rename or move it, then rerun this cell.")
else:
    df_v4.to_csv(V4_PATH, index=False); print("Saved:", V4_PATH, f"({len(df_v4)} rows, {df_v4['party'].nunique()} parties)")

### Two more parties - Jsme Praha a Spojen8 levice

In [ ]:
# S0-2: add leaflet furniture to NOISE, rebuild jsme_praha, never invent headline text
NOISE["jsme_praha"] = [r"VOLBY 9\.–10\. října 2026.*", r"č\. ?\d{1,2}", r"www\.jsmepraha\.cz.*", r".*jsme\.team/.*"]

jp = to_df(mark_noise(read_any("jsme_praha", FILES["jsme_praha"])), "jsme_praha")
empty = jp["headline"].isna() | jp["headline"].eq("")
jp.loc[empty, "headline"] = "(bez nadpisu)"          # explicit marker, not made-up text
jp.loc[empty & (jp["number_of_words"] < P["MIN_WORDS"]), "is_noise"] = True
dfs["jsme_praha"] = jp
print("jsme_praha:", len(jp), "passages | untitled:", int(empty.sum()), "| flagged noise:", int(jp.is_noise.sum()))

### Corpus V5 Compilation

In [ ]:
# ============ S0: corpus_v5 = corpus_v4 (unchanged) + spojena_levice + jsme_praha ============
FAILS.clear()
V4_PATH, V5_PATH = (os.path.join(CORPUS_DIR, f"corpus_{v}.csv") for v in ("v4", "v5"))
DROP_PATH = os.path.join(CORPUS_DIR, "corpus_v5_dropped.csv")
NEW_PARTIES = ["spojena_levice", "jsme_praha"]

# ---- 1. what counts as furniture / contact data (not programme content) ----
EMAIL = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
FURN = re.compile(rf"{EMAIL}"                                  # e-mail addresses
                  r"|(?:přidejte se a podpořte nás:?\s*)?"     # footer lead-in
                  r"(?:https?://|www\.)\S+"                    # any URL starting with http(s):// or www.
                  r"|\bjsme\.team\S*"                          # bare jsme.team/... links
                  r"|VOLBY 9\.–10\. října 2026"                # leaflet header
                  r"|\bNEPRODEJNÉ\b"                           # print-run label
                  r"|\bč\. ?\d{1,2}\b", re.I)                  # issue number "č. 16"
FURN_CHECK = rf"NEPRODEJNÉ|jsme\.team|jsmepraha\.cz|{EMAIL}"
PUZZLE = r"křížovk|luštitel|Tajenku"

def clean_str(s):
    return re.sub(r"\s+", " ", FURN.sub(" ", str(s))).strip(" |,;")

# ---- 2. rebuild the two new parties fresh from the source files ----
new_parts = []
for p in NEW_PARTIES:
    d = to_df(mark_noise(read_any(p, FILES[p])), p)
    d["text"] = d["text"].map(clean_str)
    d["headline"] = d["headline"].fillna("").map(clean_str)
    d["headline"] = d["headline"].replace("", "(bez nadpisu)")      # explicit marker, no invented text
    d["number_of_words"] = d["text"].map(n_words)
    d["is_noise"] = d["number_of_words"] < P["MIN_WORDS"]
    d["text_hash"] = d["text"].map(lambda t: hashlib.sha1(t.encode("utf-8")).hexdigest()[:12])
    new_parts.append(d)
new = pd.concat(new_parts, ignore_index=True)
new = new[new["number_of_words"] > 0]                               # passages that were only furniture

# ---- 3. drop leaflet puzzle passages (explicit, logged, bounded) ----
puzzle = new["text"].str.contains(PUZZLE, regex=True)
print("dropping puzzle passages:\n", new.loc[puzzle, ["passage_id", "number_of_words"]]
      .assign(text=new.loc[puzzle, "text"].str[:80]).to_string())
assert 1 <= puzzle.sum() <= 3, f"puzzle pattern matched {int(puzzle.sum())} passages, check before dropping"
dropped_log = new.loc[puzzle, ["passage_id", "party", "text_hash"]].assign(reason="leaflet puzzle, not programme")
new = new[~puzzle].reset_index(drop=True)

# ---- 4. combine with the frozen v4 ----
v4 = pd.read_csv(V4_PATH, dtype={"passage_id": str, "text_hash": str}); v4["is_noise"] = to_bool(v4["is_noise"])
df_v5 = pd.concat([v4, new.reindex(columns=v4.columns)], ignore_index=True)

# ---- 5. checks ----
report(df_v5["passage_id"].is_unique, "passage_id unique")
report(df_v5["text_hash"].is_unique, "text_hash unique")
report(df_v5["party"].nunique() == 10, f"10 parties (found {df_v5['party'].nunique()})")
report(set(NEW_PARTIES) <= set(df_v5["party"]), "both new parties present")
report(not (new["headline"].isna() | new["headline"].eq("")).any(), "no empty headline in new parties")
bad = df_v5[df_v5["text"].str.contains(FURN_CHECK, regex=True) | df_v5["headline"].fillna("").str.contains(FURN_CHECK, regex=True)]
report(bad.empty, f"no furniture or e-mail in any party (offending rows: {len(bad)})")
if not bad.empty:                                                    # show where, so we can decide
    for _, r in bad.iterrows():
        s = r["text"] if re.search(FURN_CHECK, r["text"]) else r["headline"]; m = re.search(FURN_CHECK, s)
        print(f"  [{r.passage_id}] ...{s[max(0, m.start()-60):m.end()+40]!r}")
report(set(zip(v4.passage_id, v4.text_hash)) <= set(zip(df_v5.passage_id, df_v5.text_hash)), "all v4 rows unchanged")
assert not FAILS, FAILS                                              # nothing is saved if any check failed

# ---- 6. freeze (never overwrite) ----
if os.path.exists(V5_PATH):
    cur = pd.read_csv(V5_PATH, dtype={"passage_id": str, "text_hash": str})
    same = set(zip(cur.passage_id, cur.text_hash)) == set(zip(df_v5.passage_id, df_v5.text_hash))
    print("v5 exists:", "matches, not overwritten" if same else "DIFFERS, rename it and rerun")
else:
    df_v5.to_csv(V5_PATH, index=False); dropped_log.to_csv(DROP_PATH, index=False)
    print("saved", os.path.basename(V5_PATH), len(df_v5), "rows | dropped log:", os.path.basename(DROP_PATH))
print(df_v5.groupby("party").agg(passages=("passage_id", "count"), words=("number_of_words", "sum"), noise=("is_noise", "sum")))

# CAN JUST RUN FROM HERE
Run cells 1–3 first, then everything from here down (Runtime → Run after). The trace cells skip themselves; the app starts in D2 and is checked in T1.


In [ ]:
# R0 | Starting from here? Cells 1-3 must have run first. Defines the helpers of cell 46 only if they are missing.
if "FAILS" not in globals():
    FAILS = []
    def report(ok, msg):
        print(("PASS  " if ok else "FAIL  ") + msg)
        if not ok: FAILS.append(msg)
if "l2norm" not in globals():
    def l2norm(M):
        return M / np.linalg.norm(M, axis=1, keepdims=True)
print("helpers ready")


# Active Corpus version Checker

Loading the newest version of the corpus and checking which version is used. This is important as we aim to be running on the latest Corpus availible.

Checks the loaded corpus, adds topic labels to every row, and makes df_clust (all rows without noise)

In [ ]:
# ---- Load the newest corpus (the version comes from the file name) ----
#FAILS.clear()
def to_bool(s):
    # CSV round trip can turn booleans into text; this avoids a silently wrong noise filter
    return s.astype(str).str.lower().isin(["true", "1"])
found = [f for f in glob.glob(os.path.join(CORPUS_DIR, "corpus_v*.csv"))
         if re.fullmatch(r"corpus_v\d+\.csv", os.path.basename(f))]
assert found, f"No corpus_v*.csv in {CORPUS_DIR}"
CORPUS_PATH = max(found, key=lambda f: int(re.search(r"v(\d+)", os.path.basename(f)).group(1)))
P["corpus_version"] = re.search(r"corpus_(v\d+)", os.path.basename(CORPUS_PATH)).group(1)
print(CORPUS_PATH)

df_all = pd.read_csv(CORPUS_PATH, dtype={"passage_id": str, "text_hash": str})
df_all["is_noise"] = to_bool(df_all["is_noise"])      # CSV can turn booleans into text
REQUIRED = ["passage_id", "party", "page", "heading_1", "headline", "text",
            "number_of_words", "item_type", "is_noise", "text_hash"]   # topic_general comes from T0

print("Corpus:", os.path.basename(CORPUS_PATH), "| version:", P["corpus_version"])
print(f"Total: {len(df_all)} | noise: {int(df_all['is_noise'].sum())} | to cluster: {int((~df_all['is_noise']).sum())}")

## Check the claims

For each passage: its variables, the heading rule (R1–R10) that produced its headline, and the blocks merged into it.
A fixed seed gives the same 5 rows on every run.

In [ ]:
if "df" in globals() and "all_blocks" in globals():
    # Trace | For each sampled passage: its variables, the rule that made the headline, and the blocks it merged
    import textwrap
    bl = pd.DataFrame(all_blocks).set_index('bid')
    SEED = 42

    def trace(r):
        print('=' * 100)
        for c in ['passage_id', 'party', 'source_doc', 'page', 'source', 'heading_1', 'headline',
                  'number_of_words']:
            print(f'{c:16}: {r[c]}')
        hb = r['headline_bid']
        if hb: print(f'{"headline rule":16}: {bl.loc[hb, "rule"]}   (block {hb}, font ratio {bl.loc[hb, "size_ratio"]})')
        print(f'{"merged blocks":16}: {len(r["bids"])}')
        for bid in r['bids']:
            x = bl.loc[bid]
            print(f'   [{bid}] {x["rule"]:34} font={x["size_ratio"]}  "{x["text"][:60]}..."')
        print(textwrap.fill(r['text'], 100, initial_indent='  ', subsequent_indent='  '))

    for _, r in df.sample(5, random_state=SEED).iterrows(): trace(r)
else:
    print('Trace skipped: needs the full rebuild (cell E3+F). Not needed for the app.')


Same check for each party.

In [ ]:
if "df" in globals() and "cols" in globals():
    # Check | One random passage per party (8 rows)
    import textwrap
    for _, r in df.groupby('party').sample(1, random_state=SEED).iterrows():
        print('=' * 100)
        for c in cols[:-1]:
            v = str(r[c])
            print(f'{c:16}: {v[:16] + "..." if c == "sha256" else v}')
        print('text:')
        print(textwrap.fill(r['text'], width=100, initial_indent='  ', subsequent_indent='  '))
else:
    print('Check skipped: needs the full rebuild (cells E3+F, F2). Not needed for the app.')


# Topic assesment

## Baseline: Rule-based system

In [ ]:
# ---- T0: topic baseline. Defined here, applied to every row in cell 4 ----
TOPICS = {
 'bezpečnost':                 [r'bezpeč', r'polici', r'strážn', r'kriminal', r'okrsk', r'drog', r'hazard', r'kamer'],
 'bydlení':                    [r'\bbyt', r'bydl', r'nájem', r'nájm', r'developer', r'výstavb', r'stavební'],
 'doprava':                    [r'doprav', r'\bmhd\b', r'\bmetr(o|a|u|em)\b', r'tramvaj', r'autobus', r'parkov', r'cykl', r'silnic', r'okruh'],
 'zeleň a veřejný prostor':    [r'zelen', r'\bpark(?!ov)', r'strom', r'klima', r'veřejn\w* prostor', r'odpad', r'čistot'],
 'rozpočet a transparentnost': [r'rozpoč', r'transparen', r'korupc', r'dluh', r'\bdaň', r'zakáz', r'digitaliz', r'úřad'],
 'kultura a sociální':         [r'kultur', r'sociál', r'senior', r'škol', r'zdravot', r'sport', r'rodin', r'\bdět'],
}

def topic_hits(headline, text):
    """Which stems fired, per topic: {topic: {stem: count}}. Headline hits count double."""
    h, t = headline.lower(), text.lower()
    out = {}
    for k, ps in TOPICS.items():
        hits = {p: 2 * bool(re.search(p, h)) + len(re.findall(p, t)) for p in ps}
        hits = {p: n for p, n in hits.items() if n}
        if hits: out[k] = hits
    return out

def topic(headline, text):
    score = {k: sum(v.values()) for k, v in topic_hits(headline, text).items()}
    return max(score, key=score.get) if score else 'jiné'   # dict order = TOPICS order -> same tie-break as in 01

def add_topics(df):
    """Label every row. Check: rows that already had a label (written by notebook 01) must keep it."""
    old = df['topic_general'] if 'topic_general' in df else pd.Series(np.nan, index=df.index)
    df['topic_general'] = [topic(h, t) for h, t in zip(df['headline'].fillna(''), df['text'])]
    had = old.notna()
    n_diff = int((old[had] != df.loc[had, 'topic_general']).sum())
    report(n_diff == 0, f"same label as notebook 01 on {int(had.sum())} rows that had one ({n_diff} differ)")
    report(df['topic_general'].notna().all(), f"every row has a topic ({len(df)} rows)")
    return df

In [ ]:

missing = [c for c in REQUIRED if c not in df_all.columns]
assert not missing, f"Missing columns: {missing}"
assert df_all["passage_id"].is_unique, "passage_id is not unique"
assert df_all["text_hash"].notna().all(), "text_hash has empty values"
df_all = add_topics(df_all)
df_all["is_noise"] = to_bool(df_all["is_noise"])

assert not FAILS, f"Topic check failed: {FAILS}"


df_clust = df_all[~df_all["is_noise"]].reset_index(drop=True)   # noise stays in df_all only


print(df_clust["party"].value_counts().to_string())
print("\nEmpty headlines by party:")
print(df_clust["headline"].isna().groupby(df_clust["party"]).sum().to_string())
print("\nTopics by party (passages):")
print(pd.crosstab(df_clust["topic_general"], df_clust["party"]).to_string())

### Topic charts

Now drawn on the loaded corpus version, so the corrected Praha sobě and Starostové rows are included.
The first chart shows the amount of text per topic (long programmes dominate it). The second normalises each party to 100 % and shows what each party emphasises.

In [ ]:
# Chart | Stacked bars: one bar per topic, one colour per party
import matplotlib.pyplot as plt
FIG_DIR = os.path.join(PROJECT_DIR, "data", "interim")   # same folder notebook 01 used for figures

METRIC = 'number_of_words'   # 'number_of_words' = amount of text, or 'passages' = number of passages

if METRIC == 'passages':
    tab = df_all.groupby(['topic_general', 'party']).size().unstack(fill_value=0)
else:
    tab = df_all.groupby(['topic_general', 'party'])['number_of_words'].sum().unstack(fill_value=0)

tab = tab.loc[tab.sum(axis=1).sort_values(ascending=False).index]   # biggest topic first

ax = tab.plot(kind='bar', stacked=True, figsize=(11, 6), colormap='tab10', width=0.8, edgecolor='white')
ax.set_xlabel('topic_general')
ax.set_ylabel('words' if METRIC == 'number_of_words' else 'passages')
ax.set_title(f'Programme content per topic and party ({METRIC})')
ax.legend(title='party', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'topics_by_party_{METRIC}_{P["corpus_version"]}.png'), dpi=150)
plt.show()
print(tab)

In [ ]:
# Chart | Per party: share of its words per topic (removes the length effect)
share = df_all.groupby(['party', 'topic_general'])['number_of_words'].sum().unstack(fill_value=0)
share = share.div(share.sum(axis=1), axis=0) * 100

ax = share.plot(kind='bar', stacked=True, figsize=(11, 6), colormap='tab10', width=0.8, edgecolor='white')
ax.set_ylabel('% of party programme (words)')
ax.set_title('Topic emphasis per party')
ax.legend(title='topic', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'topic_share_per_party_{P["corpus_version"]}.png'), dpi=150)
plt.show()

# Embedding

Cells below embeds all clustering passages in batches of 32 and saves them to Drive, so reruns only call the API for missing passages.

Model used:
| Model Type | Models | Size | Layers | Sequence Length | Embedding Dimension | MRL Support | Instruction Aware |
|---|---|---|---|---|---|---|---|
| Text Embedding | Qwen3-Embedding-4B | 4B | 36 | 32K | 2560 | Yes | Yes |

In [ ]:
# ---- Which corpus is loaded
print("E0 loaded:", CORPUS_PATH, "| P version:", P["corpus_version"])

In [ ]:
# ---- E1: Smoke test (embedding API) -----------------------------------------
from google.colab import userdata
from openai import OpenAI

API_KEY = userdata.get("LITELLM_API_KEY")  # from Colab secrets, never typed into the notebook

# Two related sentences (housing) and one unrelated control (park)
SENT_A = "Město musí podpořit dostupné bydlení pro mladé rodiny."
SENT_B = "Radnice by měla pomoci s levným nájemním bydlením."
SENT_C = "Na Vinohradech se má obnovit park u školy."

def try_embed(base_url):
    client = OpenAI(base_url=base_url, api_key=API_KEY)
    resp = client.embeddings.create(model=P["embed_model"], input=[SENT_A, SENT_B, SENT_C])
    return np.array([d.embedding for d in resp.data])

# The OpenAI client appends /embeddings to base_url, so the standard form ends in /v1.
# Try that first, then the bare root as a fallback.
candidates = [P["base_url"].rstrip("/") + "/v1", P["base_url"].rstrip("/")]
vecs, working_url = None, None
for url in candidates:
    try:
        vecs = try_embed(url)
        working_url = url
        break
    except Exception as e:
        print(f"FAILED {url}: {type(e).__name__}: {str(e)[:200]}")

assert working_url is not None, "No URL worked. Check the proxy address and the API key."
P["base_url"] = working_url  # store the working one for later cells
print("Working base_url:", working_url)

# L2-normalise, so the dot product equals cosine similarity
def l2norm(M):
    return M / np.linalg.norm(M, axis=1, keepdims=True)

V = l2norm(vecs)
sim = V @ V.T
print("Vector dimension:", V.shape[1])
print(f"cos(A, B) related: {sim[0, 1]:.3f}")
print(f"cos(A, C) unrelated: {sim[0, 2]:.3f}")
assert not np.isnan(V).any(), "NaN in embeddings"
assert sim[0, 1] > sim[0, 2], "Related pair should score higher than the unrelated pair"

The embedding itself. We used access to university infrastructure. At first we test the embedding model and then we use it to embedd the active corpus.

We used embedigns on whole programme claims that we have extracted earlier using rule based system.

In [ ]:
# ---- 6. E2: embed passages, with a disk cache per model and corpus version ----
from openai import APIError, RateLimitError, APIConnectionError, AuthenticationError

assert P["corpus_version"], "Run cell 4 first"
assert P["base_url"].endswith("/v1"), "Run cell 5 first"
client = OpenAI(base_url=P["base_url"], api_key=API_KEY)

MODEL, CORPUS_VER = P["embed_model"], P["corpus_version"]
SLUG = MODEL.replace(":", "_")                      # "qwen3-embedding:4b" -> "qwen3-embedding_4b" (no ":" in file names)
EMB_PATH = os.path.join(EMB_DIR, f"emb_{SLUG}_{CORPUS_VER}.npy")
IDX_PATH = os.path.join(EMB_DIR, f"emb_{SLUG}_{CORPUS_VER}_index.csv")
print("Embedding cache:", os.path.basename(EMB_PATH), "| exists:", os.path.exists(EMB_PATH))

def embed_batch(texts, prefix=""):      # prefix is for queries (stage 04); passages use none
    for attempt in range(5):
        try:
            r = client.embeddings.create(model=MODEL, input=[prefix + t for t in texts])
            return [d.embedding for d in r.data]
        except AuthenticationError:
            raise
        except (RateLimitError, APIConnectionError, APIError) as e:
            wait = 2 ** attempt
            print(f"  {type(e).__name__}, retrying in {wait}s"); time.sleep(wait)
    raise RuntimeError("Embedding failed after 5 attempts")

if os.path.exists(EMB_PATH) and os.path.exists(IDX_PATH):
    cache_vecs = np.load(EMB_PATH)
    index = pd.read_csv(IDX_PATH, dtype={"text_hash": str})["text_hash"].tolist()
    assert len(index) == len(cache_vecs), "Cache index and vectors disagree"
else:
    cache_vecs, index = np.empty((0, 0), dtype=np.float32), []

todo = df_clust[["text_hash", "text"]].drop_duplicates("text_hash")
todo = todo[~todo["text_hash"].isin(index)]
print(f"Passages: {len(df_clust)} | cached texts: {len(index)} | to embed: {len(todo)}")

B = P["embed_batch"]
hashes, texts = todo["text_hash"].tolist(), todo["text"].tolist()
for start in range(0, len(texts), B):
    vecs = l2norm(np.array(embed_batch(texts[start:start + B]), dtype=np.float32))
    cache_vecs = vecs if cache_vecs.size == 0 else np.vstack([cache_vecs, vecs])
    index += hashes[start:start + B]
    np.save(EMB_PATH, cache_vecs)                       # save every batch: a crash loses at most one
    pd.DataFrame({"text_hash": index, "row": range(len(index))}).to_csv(IDX_PATH, index=False)
    print(f"  embedded {min(start + B, len(texts))}/{len(texts)}")

row_of = {h: i for i, h in enumerate(index)}
E_qwen = cache_vecs[[row_of[h] for h in df_clust["text_hash"]]]   # aligned with df_clust
print("Embedding matrix:", E_qwen.shape)

In [ ]:
# ---- 7. E3 checks: shape, NaN, norms, neighbours of fixed queries ----
assert E_qwen.shape[0] == len(df_clust), "One vector per passage is required"
assert not np.isnan(E_qwen).any(), "NaN in embeddings"
assert np.allclose(np.linalg.norm(E_qwen, axis=1), 1.0, atol=1e-3), "Vectors are not unit length"
print("Corpus:", P["corpus_version"], "| matrix:", E_qwen.shape, "| norms OK")
print("Max words in a passage:", int(df_clust["number_of_words"].max()))

S = E_qwen @ E_qwen.T
np.fill_diagonal(S, -1)
head = df_clust["headline"].fillna("")        # empty headline -> "" so string tests never return NaN

def find(mask, name):
    idx = df_clust.index[mask]
    assert len(idx), f"Query passage not found: {name}"
    return idx[0]

queries = {
    "prahasobe housing (item 18)": find(head.str.contains("MĚSTSKÉ POZEMKY VYUŽIJEME", regex=False)
                                        & (df_clust["party"] == "prahasobe"), "p18"),
    "ano 7.6": find(head.str.startswith("7.6 Doktor"), "ano 7.6"),
    "starostove Bytová politika": find((df_clust["party"] == "starostove") & (head == "Bytová politika"), "starostove"),
}
for name, i in queries.items():
    print(f"\nQUERY {name}  [{df_clust.at[i, 'passage_id']}]")
    for j in np.argsort(-S[i])[:3]:
        print(f"  {S[i, j]:.3f} [{df_clust.at[j, 'party']}] {head[j][:80]}")

# Clustering
Plan
- C1: UMAP to 10 dimensions (cosine, min_dist 0.0, seed 42), then HDBSCAN. One run, no tuning yet.
- C2: Cluster sizes, noise share, party-by-cluster table, and ARI against party and against topic, both computed on the **same rows** (clustered passages only). If ARI(party) is higher than ARI(topic), the clusters are mostly about who wrote the text. That is a finding to record, not a bug to hide.
- C3: The passages closest to each cluster centre (centroid_reps = 5), plus a 3-D plot of the clusters coloured by party and by cluster. The plot uses the same `min_dist` as clustering (`P["umap_min_dist"]` = 0.0), but it is a separate 3-D UMAP fit, so it only illustrates the 10-D space and is not identical to it.
Then: tune only if C2 shows a problem, one parameter at a time, and log each change with before/after numbers.

In [ ]:
# ---- C1: reduce and cluster
import umap
from sklearn.cluster import HDBSCAN

reducer = umap.UMAP(n_components=P["umap_dims_cluster"], n_neighbors=P["umap_n_neighbors"],
                    min_dist=P["umap_min_dist"], metric="cosine", random_state=SEED)
Z = reducer.fit_transform(E_qwen)

labels = HDBSCAN(min_cluster_size=P["hdbscan_min_cluster_size"],
                 min_samples=P["hdbscan_min_samples"]).fit_predict(Z)
df_clust["cluster"] = labels
print("clusters:", len(set(labels) - {-1}), "| noise share:", round((labels == -1).mean(), 3))

In [ ]:
# ---- C2: what do the clusters follow, party or topic? ----
from sklearn.metrics import adjusted_rand_score

print(pd.crosstab(df_clust["cluster"], df_clust["party"]).to_string())

m = labels != -1                                 # the SAME rows for both scores: clustered passages only
assert df_clust.loc[m, "topic_general"].notna().all(), "Run cell 4 first (T0 labels every row)"
print(f"\nRows used: {int(m.sum())} of {len(df_clust)} (unclustered, left out: {int((~m).sum())})")
print("ARI vs party:", round(adjusted_rand_score(df_clust.loc[m, "party"], labels[m]), 3))
print("ARI vs topic:", round(adjusted_rand_score(df_clust.loc[m, "topic_general"], labels[m]), 3))

print("\nCluster x topic (clustered rows):")
print(pd.crosstab(df_clust.loc[m, "cluster"], df_clust.loc[m, "topic_general"]).to_string())

In [ ]:
# ---- C3 (3-D view): Plotly Interactive 3-D Scatter Plot with custom hovered fields ----
import umap
import plotly.express as px
import pandas as pd
import numpy as np

# Compute 3D UMAP coordinates
# min_dist from the config (0.0), the same value as clustering in C1
reducer_3d = umap.UMAP(n_components=3, n_neighbors=P["umap_n_neighbors"], min_dist=P["umap_min_dist"],
                       metric="cosine", random_state=SEED)
Z3 = reducer_3d.fit_transform(E_qwen)

# Create a temporary DataFrame for plotting containing all specified parameters except text
plot_df = pd.DataFrame({
    "UMAP 1": Z3[:, 0],
    "UMAP 2": Z3[:, 1],
    "UMAP 3": Z3[:, 2],
    "passage_id": df_clust["passage_id"],
    "party": df_clust["party"],
    "source_doc": df_clust["source_doc"],
    "headline": df_clust["headline"].fillna(""),
    "number_of_words": df_clust["number_of_words"],
    "topic_general": df_clust["topic_general"],
    "cluster": labels.astype(str)
})

hover_cfg = {
    "passage_id": True,
    "party": True,
    "source_doc": True,
    "headline": True,
    "number_of_words": True,
    "topic_general": True,
    "UMAP 1": False,
    "UMAP 2": False,
    "UMAP 3": False
}

fig_party = px.scatter_3d(
    plot_df, x="UMAP 1", y="UMAP 2", z="UMAP 3",
    color="party",
    hover_name="headline",
    hover_data=hover_cfg,
    title="3D UMAP colored by Party (Interactive)",
    opacity=0.8
)
fig_party.update_traces(marker=dict(size=4))
fig_party.show()

fig_cluster = px.scatter_3d(
    plot_df, x="UMAP 1", y="UMAP 2", z="UMAP 3",
    color="cluster",
    hover_name="headline",
    hover_data=hover_cfg,
    title="3D UMAP colored by Cluster (Interactive)",
    opacity=0.8
)
fig_cluster.update_traces(marker=dict(size=4))
fig_cluster.show()

In [ ]:
# ---- C3: representatives per cluster and a 2-D view ----
import matplotlib.pyplot as plt

for c in sorted(set(labels) - {-1}):
    idx = np.where(labels == c)[0]
    sims = E_qwen[idx] @ E_qwen[idx].mean(axis=0)
    print(f"\nCLUSTER {c} ({len(idx)} passages)")
    for j in idx[np.argsort(-sims)[:P["centroid_reps"]]]:
        print(f"  [{df_clust.at[j, 'party']}] {str(df_clust.at[j, 'headline'])[:60]} | {df_clust.at[j, 'text'][:90]}")

## Cluster description

TF-IDF Baseline and comparison with Clusters made using embeddings

# NER (removed from the pipeline)
Tried in A1–A6: GLiNER entities + an alias dictionary (MHD, DPP, PID, metro) as a boost on top of dense retrieval.

**Conclusion (kept from the original section):** On 12 hand-labelled queries, dense retrieval ranks a relevant passage first in most cases (MRR 0.92 dev, 0.88 test). An entity-overlap boost gave no measurable gain at small weights and degraded precision at w ≥ 0.1. NER is therefore not used anywhere in the pipeline or the app.

The code cells were removed on 2026-11-25 (see the Failure Journal). Kept on purpose:
- `ner_retrieval_gold_v5.csv`: the Stage B gold file read by B0. The name is historical; do not rename it.
- `simplemma` is now installed in cell 2 (B1 needs it; the NER cell used to install it).


# Retrieval

## Test queries

those queries are used trought the whole testing.



In [ ]:
# ---- B0: Stage B test queries (frozen). Gold passage ids are read from ner_retrieval_gold_v5.csv ----
# The 12 queries of A6, written here with correct diacritics. Never edit after seeing retrieval results.
QUERIES = pd.DataFrame([
    ("q00", "dev",  "Praha by měla dostavět metro D."),
    ("q01", "dev",  "Dopravní podnik by měl zdražit jízdné."),
    ("q02", "dev",  "Město má stavět více městských bytů."),
    ("q03", "dev",  "Praha má zpřísnit krátkodobé pronájmy typu Airbnb."),
    ("q04", "dev",  "Praha má zvýšit počet strážníků v ulicích."),
    ("q05", "dev",  "Město má vysazovat více stromů na sídlištích."),
    ("q06", "test", "Praha by měla posílit MHD ve špičce."),
    ("q07", "test", "Magistrát má zveřejňovat všechny smlouvy městských firem."),
    ("q08", "test", "Město má dotovat kulturu z veřejných peněz."),
    ("q09", "test", "Parkování pro návštěvníky má být dražší."),
    ("q10", "test", "Pražský okruh se má dostavět."),
    ("q11", "test", "Školy mají otevřít hřiště veřejnosti."),
], columns=["query_id", "split", "query"])

# Gold file = Czech Excel export: ";" separated, cp1250 (reading it as cp1252 gave "mìla" instead of "měla")
GOLD_PATH = os.path.join(CORPUS_DIR, "ner_retrieval_gold_v5.csv")
for enc in ("utf-8-sig", "cp1250"):
    try: gold = pd.read_csv(GOLD_PATH, sep=";", dtype=str, encoding=enc).fillna(""); break
    except UnicodeDecodeError: pass
gold.columns = [c.strip() for c in gold.columns]
ids = lambda s: [p for p in re.split(r"[;,\s]+", s) if p]
gold["agree"], gold["against"] = gold["relevant_ids"].map(ids), gold["relevant_ids PROTI"].map(ids)
gold["query"] = gold["query"].str.strip()

# join by query text (the gold file's query_id column is not reliable: first row reads "c")
Q = QUERIES.merge(gold[["query", "agree", "against"]], on="query", how="left", validate="one_to_one")
assert Q["agree"].notna().all(), f"query text not found in the gold file: {Q.loc[Q['agree'].isna(), 'query'].tolist()}"
Q["rel"] = [list(dict.fromkeys(a + b)) for a, b in zip(Q["agree"], Q["against"])]   # retrieval truth = agree + against

# checks: same counts as the A6 output, ids exist among non-noise passages, no passage in both lists
EXPECTED = [(7, 0), (2, 4), (16, 1), (9, 1), (3, 0), (5, 0), (5, 0), (6, 0), (6, 1), (5, 1), (7, 0), (4, 0)]
got = list(zip(Q["agree"].map(len), Q["against"].map(len)))
assert got == EXPECTED, f"gold counts changed since A6: {got}"
pid_row = {p: i for i, p in enumerate(df_clust["passage_id"])}
bad = sorted({p for r in Q["rel"] for p in r if p not in pid_row})
assert not bad, f"unknown or noise passage_ids: {bad[:8]}"
both = [(q, sorted(set(a) & set(b))) for q, a, b in zip(Q["query_id"], Q["agree"], Q["against"]) if set(a) & set(b)]
if both: print("WARNING: passage in both agree and against:", both)

print(Q.assign(n_agree=Q["agree"].map(len), n_against=Q["against"].map(len))
       [["query_id", "split", "query", "n_agree", "n_against"]].to_string(index=False))
print("\nqueries per split:", Q["split"].value_counts().to_dict(), "| gold passages:", sum(map(len, Q["rel"])))

## TF-IDF retrieval

In [ ]:
# B1 | Stage B config + TF-IDF matrices (fit on the 696 passages only; queries are only transformed)
import hashlib, pickle, simplemma
from sklearn.feature_extraction.text import TfidfVectorizer

K_PARTY = 10           # passages per party passed on to Stage C (fixed decision: top 10 per party)
K_HIT = 3              # depth of the party-level metric (small parties have only 15-16 passages)
QUERY = "Praha by měla dostavět metro D."   # one-query demo; the evaluation uses all of Q from B0
TEXTS = df_clust["text"].tolist()
PARTY = df_clust["party"].to_numpy()

LEMMA_FIX = {"práh": "praha"}               # simplemma turns "Prahy" into "práh" (E024)
def lemmas(s): return [LEMMA_FIX.get(x, x) for x in
                       (simplemma.lemmatize(w, lang="cs").lower() for w in re.findall(r"\w+", s.lower()))]

TFIDF = {"tfidf_char":  TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=P["tfidf_min_df"]),
         "tfidf_lemma": TfidfVectorizer(analyzer=lemmas, sublinear_tf=True, min_df=P["tfidf_min_df"])}
TF_X = {name: vec.fit_transform(TEXTS) for name, vec in TFIDF.items()}
for name, M in TF_X.items(): print(f"{name}: {M.shape[0]} passages x {M.shape[1]} features")
print("lemma check:", lemmas("Prahy Praze metra stanicí"))

In [ ]:
# B2 | Scores for one query + per-party top-K (never a global top-k: a long programme would crowd out short ones)
def tfidf_scores(q, name):                  # cosine: TF-IDF rows are already unit length
    return (TF_X[name] @ TFIDF[name].transform([q]).T).toarray().ravel()

def per_party(scores, k=K_PARTY):
    rows = []
    for party in sorted(set(PARTY)):
        idx = np.where(PARTY == party)[0]
        top = idx[np.argsort(-scores[idx], kind="stable")[:k]]
        rows += [dict(party=party, passage_id=df_clust.at[i, "passage_id"], rank_in_party=r + 1, score=float(scores[i]))
                 for r, i in enumerate(top)]
    return pd.DataFrame(rows)

### TF-IDF matrix of the corpus

### Retrieval

In [ ]:
# B3 | Dense scores: query embedded with the Qwen3 instruction prefix, cached on disk (one API call per new query)
INSTR = "Instruct: Given a Czech statement about city politics, retrieve programme passages that discuss it\nQuery: "
QCACHE_PATH = EMB_DIR / f"query_cache_{SLUG}.pkl"
qcache = pickle.load(open(QCACHE_PATH, "rb")) if QCACHE_PATH.exists() else {}
def dense_scores(q):
    key = hashlib.sha1(f"{MODEL}|{INSTR}|{q}".encode("utf-8")).hexdigest()
    if key not in qcache:
        qcache[key] = l2norm(np.array(embed_batch([q], prefix=INSTR), dtype=np.float32))[0]
        pickle.dump(qcache, open(QCACHE_PATH, "wb"))
    return E_qwen @ qcache[key]

SCORERS = {"dense": dense_scores,
           "tfidf_char": lambda q: tfidf_scores(q, "tfidf_char"),
           "tfidf_lemma": lambda q: tfidf_scores(q, "tfidf_lemma")}

for name, f in SCORERS.items():             # demo: best score and top-K mean per party for one query
    t = per_party(f(QUERY))
    print(f"\n{name} | {QUERY}")
    print(t.groupby("party")["score"].agg(top1="max", topk_mean="mean").round(3).sort_values("top1", ascending=False).to_string())

In [ ]:
# B4 | Evaluation on the 12 frozen queries (Q from B0). Nothing is tuned; dev picks the retriever, test is reported.
rows = []
for _, g in Q.iterrows():
    rel = np.array([pid_row[p] for p in g.rel]); gold_parties = set(PARTY[rel])
    for name, f in SCORERS.items():
        s = f(g["query"]); order = np.argsort(-s, kind="stable")
        first = np.where(np.isin(order, rel))[0][0]
        top = per_party(s, k=K_HIT); hit = set(top.loc[top["passage_id"].isin(g.rel), "party"])
        rows.append(dict(split=g.split, query_id=g.query_id, retriever=name, rr=1 / (first + 1),
                         p5=float(np.isin(order[:5], rel).mean()), r10=float(np.isin(rel, order[:10]).mean()),
                         party_hit=len(hit & gold_parties) / len(gold_parties)))
RES = pd.DataFrame(rows)
print(RES.groupby(["split", "retriever"])[["rr", "p5", "r10", "party_hit"]].mean().round(3).to_string())
print("\nMRR per query (where the retrievers disagree):")
print(RES.pivot(index="query_id", columns="retriever", values="rr").round(2).to_string())
dev = RES[RES["split"] == "dev"].groupby("retriever")[["party_hit", "rr"]].mean()
BEST = dev.sort_values(["party_hit", "rr"], ascending=False).index[0]
print("\nretriever for Stage C (rule fixed in advance: dev party_hit, then dev MRR):", BEST)
RES.to_csv(os.path.join(CORPUS_DIR, "stage_b_eval_v5.csv"), index=False)

In [ ]:
# B5 | Error examples (dev only): when the top hit is not relevant, what came first and where the first gold passage was
for name, f in SCORERS.items():
    for _, g in Q[Q["split"] == "dev"].iterrows():
        s = f(g["query"]); order = np.argsort(-s, kind="stable"); rel = {pid_row[p] for p in g.rel}
        if order[0] in rel: continue
        first = next(r for r, i in enumerate(order) if i in rel); top = df_clust.iloc[order[0]]
        print(f"[{name}] {g.query_id} {g.query}\n   top-1 {top.passage_id} ({s[order[0]]:.3f}): {top.text[:110]}\n   first gold passage at rank {first + 1}")

In [ ]:
# B6 | Retrieval output in the data-contract format (PROJECT_STATE section 3), for Stage C and the ranking half
out = pd.concat([per_party(SCORERS[BEST](g["query"])).assign(thesis_id=g.query_id, retriever=BEST) for _, g in Q.iterrows()])
OUT_PATH = os.path.join(CORPUS_DIR, f"retrieval_{BEST}_v5.csv")
out[["thesis_id", "party", "passage_id", "rank_in_party", "retriever", "score"]].to_csv(OUT_PATH, index=False)
print(len(out), "rows ->", OUT_PATH, "| rows per thesis:", sorted(out.groupby("thesis_id").size().unique()))

## Comparison

# LLM Stance

LLM should somehow consider the relevancy of returned programme stances and rank the stances for how much they agree with the queries. The result is the ranked programme stances and score for relevancy (agree +1, disagree -1) Not relevant are zero.


In [ ]:
# C0 | How deep must Stage C read? Share of gold passages inside each party's top-k (dev only, from B6 output)
ret = pd.read_csv(OUT_PATH)
rank_of = {(r.thesis_id, r.passage_id): r.rank_in_party for r in ret.itertuples()}
gp = pd.DataFrame([(g.query_id, g.split, p, "AGREE" if p in g.agree else "DISAGREE") for _, g in Q.iterrows() for p in g.rel],
                  columns=["thesis_id", "split", "passage_id", "gold"])
gp["rank_in_party"] = [rank_of.get(k, 99) for k in zip(gp["thesis_id"], gp["passage_id"])]
dev = gp[gp["split"] == "dev"]
print("dev gold pairs:", len(dev), "| share inside the party's top-k:", {k: round((dev["rank_in_party"] <= k).mean(), 3) for k in (3, 5, 10)})

In [ ]:
# C0b | Negatives for the stance gold: non-gold passages from each party's top-K, labelled by 2 annotators BEFORE any LLM run
K_STANCE = 5           # set from the C0 dev numbers
N_PER_Q = 5            # 12 queries x 5 = 60 rows, all labelled by both annotators (kappa needs >= 50 shared)
TEXT = dict(zip(df_clust["passage_id"], df_clust["text"]))
cand = ret[ret["rank_in_party"] <= K_STANCE].merge(Q[["query_id", "query", "rel"]], left_on="thesis_id", right_on="query_id")
cand = cand[[p not in rel for p, rel in zip(cand["passage_id"], cand["rel"])]]
pick = cand.sample(frac=1, random_state=SEED).groupby("thesis_id").head(N_PER_Q).reset_index(drop=True)   # shuffled: order says nothing
sheet = pick.assign(text=pick["passage_id"].map(TEXT), label_A="", label_B="")[["thesis_id", "query", "passage_id", "text", "label_A", "label_B"]]
NEG_PATH = os.path.join(CORPUS_DIR, "stance_negatives_v5.csv")
if os.path.exists(NEG_PATH): print("exists, NOT overwritten:", NEG_PATH)
else: sheet.to_csv(NEG_PATH, index=False, encoding="utf-8-sig"); print(len(sheet), "rows ->", NEG_PATH)

In [ ]:
# C1 | LLM: check the model tag on the proxy, then one smoke call (inspect the raw answer before writing the parser)
LLM_MODEL = P["llm_model"]                          # "qwen3.5:122b"; fallback "qwen3.6:35b"
available = sorted(m.id for m in client.models.list().data)
print("qwen models on the proxy:", [m for m in available if "qwen" in m.lower()])
assert LLM_MODEL in available, f"{LLM_MODEL} not on the proxy: pick the fallback from the list above"

LLM_PARAMS = dict(temperature=0, seed=SEED, max_tokens=400)
msg = [{"role": "user", "content": 'Odpověz pouze tímto JSON a ničím jiným: {"ok": true}'}]
t0 = time.time()
try:
    r = client.chat.completions.create(model=LLM_MODEL, messages=msg, **LLM_PARAMS)
except Exception as e:                              # some proxies reject `seed`: retry without it and record that
    print("with seed failed:", type(e).__name__, str(e)[:200]); LLM_PARAMS.pop("seed")
    r = client.chat.completions.create(model=LLM_MODEL, messages=msg, **LLM_PARAMS)
m = r.choices[0].message
print(f"{time.time() - t0:.1f} s | finish_reason: {r.choices[0].finish_reason} | params used: {LLM_PARAMS}")
print("content:", repr((m.content or "")[:400]))
print("separate reasoning field:", bool(getattr(m, "reasoning_content", None)))

In [ ]:
# C1 | The planned qwen3.5:122b is not on the proxy (F025). Candidates = text-capable Qwen chat models from the list
available = sorted(m.id for m in client.models.list().data)
CANDIDATES = [m for m in ["qwen3.6-35b", "Qwen3.6-35B-A3B", "Qwen3.8-27B", "qwen3-vl:235b-a22b-instruct"] if m in available]
print("candidates:", CANDIDATES)

In [ ]:
# C2 | Judge prompt v3: relevant + stance on 5 anchored steps (-1..1) + confidence on 4 anchored steps; no party names; strict parser
PROMPT_VERSION = "v3"                       # v2 + "confidence"; v2 answers stay in the cache for comparison
SCALE = (-1.0, -0.5, 0.0, 0.5, 1.0)
CONF = (0.25, 0.5, 0.75, 1.0)               # anchored levels, not probabilities
PROMPTS = {
"cs": """Jsi pečlivý analytik volebních programů. Dostaneš tezi a úryvek z programu jedné strany.
1) "relevant": true jen tehdy, když se úryvek vyjadřuje přímo k této tezi (nestačí stejné téma). Jinak false.
2) "stance" (jen když relevant je true, jinak null), jedna z hodnot:
    1 = úryvek tezi jasně podporuje nebo slibuje přesně to;
  0.5 = podporuje ji částečně, podmíněně nebo slabě (např. "prověříme", "budeme usilovat");
    0 = vyjadřuje se k ní, ale smíšeně nebo nerozhodně;
 -0.5 = je k ní spíše skeptický nebo ji odmítá jen zčásti;
   -1 = tezi jasně odmítá nebo slibuje opak.
3) "quote": doslovná citace z úryvku (nejvýše 30 slov), která hodnocení dokládá; prázdný řetězec, když relevant je false.
4) "confidence" (jen když relevant je true, jinak null): jak jistě úryvek k tezi patří a jak jistý je postoj, jedna z hodnot:
     1 = úryvek se k tezi vyjadřuje výslovně a postoj je jednoznačný;
  0.75 = spíše jisté, ale formulace je obecnější nebo nepřímá;
   0.5 = nejisté, úryvek se tezi jen dotýká nebo jde vyložit dvojím způsobem;
  0.25 = spíše odhad.
Odpověz pouze jedním objektem JSON: {"relevant": true/false, "stance": 1/0.5/0/-0.5/-1/null, "confidence": 1/0.75/0.5/0.25/null, "quote": "..."}

Teze: <<QUERY>>
Úryvek: <<PASSAGE>>""",
"en": """You are a careful analyst of election programmes. You get a claim and a passage from one party's programme, both in Czech.
1) "relevant": true only if the passage speaks directly to this claim (the same topic is not enough). Otherwise false.
2) "stance" (only if relevant is true, otherwise null), one of:
    1 = the passage clearly supports the claim or promises exactly that;
  0.5 = supports it partly, conditionally or weakly (e.g. "we will examine", "we will strive");
    0 = speaks to it, but mixed or undecided;
 -0.5 = is rather sceptical or rejects it only in part;
   -1 = clearly rejects the claim or promises the opposite.
3) "quote": a verbatim quote from the passage (at most 30 words, in Czech, copied exactly) that supports the rating; empty string if relevant is false.
4) "confidence" (only if relevant is true, otherwise null): how surely the passage belongs to the claim and how clear the stance is, one of:
     1 = the passage speaks to the claim explicitly and the stance is unambiguous;
  0.75 = fairly sure, but the wording is more general or indirect;
   0.5 = unsure, the passage only touches the claim or can be read two ways;
  0.25 = rather a guess.
Answer with one JSON object only: {"relevant": true/false, "stance": 1/0.5/0/-0.5/-1/null, "confidence": 1/0.75/0.5/0.25/null, "quote": "..."}

Claim: <<QUERY>>
Passage: <<PASSAGE>>"""}
def make_prompt(lang, query, passage): return PROMPTS[lang].replace("<<QUERY>>", query).replace("<<PASSAGE>>", passage)

def norm(s): return re.sub(r"\s+", " ", s).strip()
def parse(raw, passage):
    txt = re.sub(r"<think>.*?</think>", "", raw or "", flags=re.S)
    objs = re.findall(r"\{[^{}]*\}", txt)
    if not objs: return None, "no JSON object"
    try: d = json.loads(objs[-1])
    except json.JSONDecodeError as e: return None, f"bad JSON: {e}"
    if not isinstance(d.get("relevant"), bool): return None, f"bad 'relevant': {d}"
    st = d.get("stance")
    if d["relevant"]:
        try: st = float(st)
        except (TypeError, ValueError): return None, f"relevant but no numeric stance: {d}"
        if st not in SCALE: return None, f"stance not on the scale: {d}"
    elif st not in (None, "null", ""): return None, f"not relevant but stance given: {d}"
    else: st = None
    c = d.get("confidence")                         # same rule as stance: on the scale when relevant, else null
    if d["relevant"]:
        try: c = float(c)
        except (TypeError, ValueError): return None, f"relevant but no numeric confidence: {d}"
        if c not in CONF: return None, f"confidence not on the scale: {d}"
    else: c = None
    d["confidence"] = c
    d["stance"], d["quote"] = st, str(d.get("quote") or "")
    d["quote_ok"] = (not d["relevant"]) or (len(d["quote"]) > 0 and norm(d["quote"]) in norm(passage))
    return d, ""

def call(model, prompt, params):
    t0 = time.time()
    r = client.with_options(timeout=180).chat.completions.create(model=model, messages=[{"role": "user", "content": prompt}], **params)
    return r.choices[0].message.content or "", r.choices[0].finish_reason, time.time() - t0

In [ ]:
# C3 | Smoke test, prompt v2: same 6 invented pairs, same rule (all valid, then most correct, then fastest)
SMOKE = [  # (query, passage, expected relevant, expected direction: +1 / -1 / 0 = not relevant)
    ("Město má stavět více městských bytů.", "Zahájíme výstavbu 5 000 obecních bytů na městských pozemcích.", True, 1),
    ("Dopravní podnik by měl zdražit jízdné.", "Ceny jízdenek zmrazíme po celé volební období.", True, -1),
    ("Dopravní podnik by měl zdražit jízdné.", "Doprava v Praze je dnes chaotická a zpožděná.", False, 0),
    ("Praha má zvýšit počet strážníků v ulicích.", "Strážníkům zvýšíme platy, aby z městské policie neodcházeli.", False, 0),
    ("Město má dotovat kulturu z veřejných peněz.", "Proč by daňoví poplatníci měli platit divadla, do kterých nechodí? Kulturu ať si zaplatí ti, kdo o ni stojí.", True, -1),
    ("Pražský okruh se má dostavět.", "Prosadíme dokončení všech chybějících úseků Pražského okruhu.", True, 1),
]
PARAMS, rows = {}, []
for model in CANDIDATES:
    p = dict(temperature=0, seed=SEED, max_tokens=1024)
    for q, psg, exp_rel, exp_dir in SMOKE:
        try: raw, fin, sec = call(model, make_prompt("cs", q, psg), p)
        except Exception as e:
            if "seed" in p:
                p.pop("seed"); print(f"{model}: seed rejected ({type(e).__name__}), running without seed")
                raw, fin, sec = call(model, make_prompt("cs", q, psg), p)
            else: raise
        d, err = parse(raw, psg)
        got_dir = 0 if not d or not d["relevant"] else int(np.sign(d["stance"]))
        rows.append(dict(model=model, item=len(rows) % len(SMOKE), valid=d is not None,
                         correct=bool(d) and d["relevant"] == exp_rel and got_dir == exp_dir,
                         stance=None if not d else d["stance"], quote_ok=bool(d) and d["quote_ok"],
                         sec=round(sec, 1), finish=fin, err=err))
    PARAMS[model] = p
SM = pd.DataFrame(rows)
summary = SM.groupby("model").agg(valid=("valid", "sum"), correct=("correct", "sum"), quote_ok=("quote_ok", "sum"),
                                  median_sec=("sec", "median"), cut_off=("finish", lambda s: (s == "length").sum()))
print(summary.to_string())
print(SM.pivot(index="item", columns="model", values="stance").to_string())          # what each model said per item
print(SM.loc[SM["err"] != "", ["model", "item", "err"]].to_string())
ok = summary[summary["valid"] == len(SMOKE)].sort_values(["correct", "median_sec"], ascending=[False, True])
LLM_MODEL, LLM_MODEL_2 = ok.index[0], (ok.index[1] if len(ok) > 1 else None)
print("judge:", LLM_MODEL, PARAMS[LLM_MODEL], "| second model:", LLM_MODEL_2)

In [ ]:
# C4 | Judge with cache + one re-ask. Prints totals only: annotators must not see per-row answers before labels are frozen
JUDGE_PATH = os.path.join(CORPUS_DIR, "stance_cache_v5.jsonl")
cache = {}
if os.path.exists(JUDGE_PATH):
    with open(JUDGE_PATH, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line); cache[r["key"]] = r
QTEXT = dict(zip(Q["query_id"], Q["query"]))
FIX = {"cs": "Tvoje odpověď nebyla platná ({err}). Odpověz znovu pouze jedním objektem JSON podle pokynů; citace musí být doslovně zkopírovaná z úryvku.",
       "en": "Your answer was not valid ({err}). Answer again with one JSON object only, as instructed; the quote must be copied verbatim from the passage."}

def ask(model, msgs, params):                       # network errors: 3 tries with a pause, then stop (the cache keeps the work)
    for t in range(3):
        try: return client.with_options(timeout=180).chat.completions.create(model=model, messages=msgs, **params).choices[0].message.content or ""
        except Exception as e:
            if t == 2: raise
            print(f"  {type(e).__name__}, retrying in {5 * (t + 1)} s"); time.sleep(5 * (t + 1))

def judge(model, lang, qid, pid):
    params = PARAMS[model]
    key = hashlib.sha1(json.dumps([model, PROMPT_VERSION, lang, qid, pid, params], sort_keys=True).encode()).hexdigest()
    if key in cache: return cache[key]
    passage, t0 = TEXT[pid], time.time()
    msgs = [{"role": "user", "content": make_prompt(lang, QTEXT[qid], passage)}]
    for attempt in (1, 2):                          # one re-ask with the reason, then give up and log
        raw = ask(model, msgs, params)
        d, err = parse(raw, passage)
        if d is not None and d["quote_ok"]: break
        err = err or "quote not verbatim"
        msgs += [{"role": "assistant", "content": raw}, {"role": "user", "content": FIX[lang].format(err=err)}]
    rec = dict(key=key, model=model, prompt_version=PROMPT_VERSION, lang=lang, thesis_id=qid, passage_id=pid,
               relevant=None if d is None else d["relevant"], stance=None if d is None else d["stance"],
               confidence=None if d is None else d["confidence"],
               quote="" if d is None else d["quote"], quote_ok=bool(d) and d["quote_ok"], attempts=attempt,
               error="" if (d is not None and d["quote_ok"]) else err, raw=raw[:2000], sec=round(time.time() - t0, 1))
    with open(JUDGE_PATH, "a", encoding="utf-8") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    cache[key] = rec
    return rec

top = ret[ret["rank_in_party"] <= K_STANCE]
gold_pairs = {(g.query_id, p) for _, g in Q.iterrows() for p in g.rel}
neg = pd.read_csv(NEG_PATH, encoding="utf-8-sig")
EVAL_PAIRS = sorted(gold_pairs | set(zip(neg["thesis_id"], neg["passage_id"])))
RUNS = [(LLM_MODEL,   "cs", sorted(set(zip(top["thesis_id"], top["passage_id"])) | set(EVAL_PAIRS))),   # pipeline + evaluation
        (LLM_MODEL_2, "cs", EVAL_PAIRS),                                                               # second model: evaluation only
        (LLM_MODEL,   "en", EVAL_PAIRS)]                                                               # English prompt: evaluation only
for model, lang, todo in RUNS:
    t0 = time.time(); recs = []
    for n, (qid, pid) in enumerate(todo, 1):
        recs.append(judge(model, lang, qid, pid))
        if n % 50 == 0: print(f"  {model} {lang}: {n}/{len(todo)} ({time.time() - t0:.0f} s)")
    R = pd.DataFrame(recs)
    print(f"{model} | {lang} | pairs {len(R)} | valid {R['relevant'].notna().mean():.3f} | re-asked {(R['attempts'] == 2).mean():.3f}"
          f" | quote_ok on relevant {R.loc[R['relevant'] == True, 'quote_ok'].mean():.3f} | relevant share {(R['relevant'] == True).mean():.3f}")
    print("  confidence levels on relevant:", R.loc[R["relevant"] == True, "confidence"].value_counts().sort_index().to_dict())

In [ ]:
# C5 | Outputs for the ranking half (pipeline rows only = each party's top K_STANCE): full, ranked (relevant only), coverage
tag = f"v5_{re.sub(r'[^A-Za-z0-9.]+', '-', LLM_MODEL)}_{PROMPT_VERSION}_cs"
J = pd.DataFrame([judge(LLM_MODEL, "cs", q, p) for q, p in zip(top["thesis_id"], top["passage_id"])])   # all from the cache
full = top.merge(J[["thesis_id", "passage_id", "relevant", "stance", "confidence", "quote", "quote_ok", "model", "prompt_version", "lang", "error"]],
                 on=["thesis_id", "passage_id"], validate="one_to_one")
ranked = full[full["relevant"] == True].sort_values(["thesis_id", "stance", "confidence", "score"], ascending=[True, False, False, False])
ranked["rank_in_thesis"] = ranked.groupby("thesis_id").cumcount() + 1
cov = full.groupby(["thesis_id", "party"]).agg(n_judged=("passage_id", "size"),
                                               n_relevant=("relevant", lambda s: int((s == True).sum())),
                                               n_failed=("relevant", lambda s: int(s.isna().sum()))).reset_index()
for name, t in [("stance_full", full), ("stance_ranked", ranked), ("coverage", cov)]:
    path = os.path.join(CORPUS_DIR, f"{name}_{tag}.csv"); t.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{name}: {len(t)} rows -> {os.path.basename(path)}")
print("parties with no relevant passage, per thesis:", cov[cov["n_relevant"] == 0].groupby("thesis_id").size().to_dict())

In [ ]:
# C6 | Dev only. Rule fixed BEFORE running: tau = highest dev macro-F1 of "v3 + confidence", ties -> smaller tau.
from sklearn.metrics import f1_score, confusion_matrix
from scipy.stats import spearmanr

neg = pd.read_csv(NEG_PATH, encoding="utf-8-sig")
fix = lambda x: "NOT_RELEVANT" if str(x).strip().upper().replace("_", " ").startswith("NOT REL") else str(x).strip().upper()
for c in ("label_A", "label_B"): neg[c] = neg[c].map(fix)        # E029: spelling fixed at read time, the file stays as labelled
print("annotator labels:", neg["label_A"].value_counts().to_dict())
NEG_DIR = {"AGREE": 1, "DISAGREE": -1, "NOT_RELEVANT": 0}
neg = neg[neg["label_A"] == neg["label_B"]]                  # gold = both annotators agree
gold = {(g.query_id, p): (1 if p in g.agree else -1) for _, g in Q.iterrows() for p in g.rel}
gold.update({(r.thesis_id, r.passage_id): NEG_DIR[r.label_A] for r in neg.itertuples()})
SPLIT = dict(zip(Q["query_id"], Q["split"]))
G = pd.DataFrame([(q, p, d, SPLIT[q]) for (q, p), d in gold.items()], columns=["thesis_id", "passage_id", "gold", "split"])

C = pd.DataFrame(cache.values())                             # every cached judgement (v2 and v3)
C = C[(C["model"] == LLM_MODEL) & (C["lang"] == "cs")]

def passage_score(r, tau=0, use_conf=True):
    """None = NO POSITION. Otherwise s = c * stance (c = 1 without confidence)."""
    if r["relevant"] != True or not r["quote_ok"]: return None
    c = r["confidence"] if use_conf else 1.0
    if c < tau: return None
    return c * r["stance"]

def scores(version, tau=0, use_conf=False):                  # {(thesis_id, passage_id): s or None}
    return {(r["thesis_id"], r["passage_id"]): passage_score(r, tau, use_conf)
            for r in C[C["prompt_version"] == version].to_dict("records")}

def pair_eval(S, split):                                     # direction per pair vs gold: macro-F1 + confusion matrix
    g = G[G["split"] == split]
    pred = [0 if S.get(k) is None else int(np.sign(S[k])) for k in zip(g["thesis_id"], g["passage_id"])]
    return f1_score(g["gold"], pred, labels=[-1, 0, 1], average="macro"), confusion_matrix(g["gold"], pred, labels=[-1, 0, 1])

PARTY_OF = dict(zip(df_clust["passage_id"], df_clust["party"]))

PARTIES_ALL = sorted(set(PARTY_OF.values()))
def claim_score(scores):
    """S_p(q) from one party's kept passage scores for one claim -> (score, flag). Ties of opposite sign = 0."""
    if not scores: return 0.0, "none"                                   # no kept passage: NO POSITION
    m = max(abs(s) for s in scores)
    signs = {np.sign(s) for s in scores if abs(s) == m}                 # signs among the strongest passages
    if 1 in signs and -1 in signs: return 0.0, "contradiction"
    return max(scores, key=abs), "ok"

def party_vec(S, q):                                                    # S_p(q) for every party, same rule as the app
    per = {p: [] for p in PARTIES_ALL}
    for pid in top.loc[top["thesis_id"] == q, "passage_id"]:
        s = S.get((q, pid))
        if s is not None: per[PARTY_OF[pid]].append(s)
    return [claim_score(per[p])[0] for p in PARTIES_ALL]
def gold_vec(q):                                             # +1 / -1 / 0 per party from the gold passages
    g = Q.set_index("query_id").loc[q]
    net = dict.fromkeys(PARTIES_ALL, 0)
    for pid in g.agree: net[PARTY_OF[pid]] += 1
    for pid in g.against: net[PARTY_OF[pid]] -= 1
    return [np.sign(net[p]) for p in PARTIES_ALL]
def rank_rho(S, split):                                      # mean Spearman, tool vs gold party direction
    return np.nanmean([spearmanr(party_vec(S, q), gold_vec(q))[0] for q in Q.loc[Q["split"] == split, "query_id"]])

# 1) Calibration: among pairs the judge calls relevant, is "confidence 1" right more often than "0.5"?
cal = G[G["split"] == "dev"].merge(C[C["prompt_version"] == "v3"], on=["thesis_id", "passage_id"])
cal = cal[cal["relevant"] == True]
cal["correct"] = (np.sign(cal["stance"]) == cal["gold"]).astype(int)
print(cal.groupby("confidence").agg(n=("correct", "size"), share_correct=("correct", "mean")).round(3).to_string())
print(f"Brier {((cal['confidence'] - cal['correct'])**2).mean():.3f} | share with confidence 1: {(cal['confidence'] == 1).mean():.3f}")

# 2) tau on dev
for tau in (0, .5, .75, 1): print(f"tau {tau}: dev macro-F1 {pair_eval(scores('v3', tau, True), 'dev')[0]:.3f}")
TAU = max((0, .5, .75, 1), key=lambda t: (round(pair_eval(scores("v3", t, True), "dev")[0], 3), -t))
TAU_PATH = os.path.join(CORPUS_DIR, "confidence_tau_v3.json")
if os.path.exists(TAU_PATH): TAU = json.load(open(TAU_PATH))["tau"]; print("frozen earlier, kept:", TAU)
else: json.dump({"tau": TAU}, open(TAU_PATH, "w")); print("frozen tau:", TAU)

# 3) Three methods on dev
METHODS = {"v2 hard": scores("v2"), "v3 hard": scores("v3"), "v3 + confidence": scores("v3", TAU, True)}
for name, S in METHODS.items(): print(f"{name:16s} dev macro-F1 {pair_eval(S, 'dev')[0]:.3f} | ranking rho {rank_rho(S, 'dev'):.3f}")

In [ ]:
# C7 | Test, ONCE, with the frozen tau. Nothing here may change TAU.
for name, S in METHODS.items():
    f1, cm = pair_eval(S, "test")
    print(f"\n{name}: test macro-F1 {f1:.3f} | ranking rho {rank_rho(S, 'test'):.3f}")
    print(pd.DataFrame(cm, index=["gold -1", "gold 0", "gold +1"], columns=["pred -1", "pred 0", "pred +1"]).to_string())

In [ ]:
# C8 | Dev error examples (read them, write a one-line diagnosis each) + bias check
cal["hard_dir"] = np.sign(cal["stance"]).astype(int)
cal["conf_dir"] = [0 if c < TAU else d for c, d in zip(cal["confidence"], cal["hard_dir"])]
for name, d in [("CONFIDENCE HELPED", cal[(cal["hard_dir"] != cal["gold"]) & (cal["conf_dir"] == cal["gold"])]),
                ("CONFIDENCE HURT", cal[(cal["hard_dir"] == cal["gold"]) & (cal["conf_dir"] != cal["gold"])]),
                ("SURE BUT WRONG", cal[(cal["confidence"] == 1) & (cal["correct"] == 0)])]:
    print(f"\n===== {name}: {len(d)}")
    for r in d.head(4).itertuples():
        print(f"[{r.thesis_id} | {r.passage_id}] gold {r.gold:+d} | stance {r.stance} conf {r.confidence}\n"
              f"  claim: {QTEXT[r.thesis_id]}\n  passage: {TEXT[r.passage_id][:300]}\n  quote: {r.quote}\n  diagnosis: ...")

pipe = top.merge(C[C["prompt_version"] == "v3"], on=["thesis_id", "passage_id"])
pipe = pipe[pipe["relevant"] == True]
B = pipe.groupby("party").agg(n_relevant=("confidence", "size"), mean_conf=("confidence", "mean"))
B["words"] = df_clust.assign(w=df_clust["text"].str.split().str.len()).groupby("party")["w"].sum()
print("\n", B.round(3).sort_values("words", ascending=False).to_string())
print(f"Spearman words vs mean confidence: {spearmanr(B['words'], B['mean_conf'])[0]:.3f} (10 parties: an indication only)")

# Evaluation

### Application Evaluation & Quality Dashboard
This cell reads frozen evaluation datasets (`stage_b_eval_v5.csv` and  cached LLM judgments) to display an overall performance report for the application.

In [ ]:
import pandas as pd
import os

eval_b_path = os.path.join(CORPUS_DIR, "stage_b_eval_v5.csv")

if os.path.exists(eval_b_path):
    df_b = pd.read_csv(eval_b_path)
    print("=== RETRIEVAL PERFORMANCE (STAGE B) ===")
    summary_b = df_b.groupby(["split", "retriever"])[["rr", "p5", "r10", "party_hit"]].mean()
    display(summary_b.round(3))
else:
    print(f"Retrieval evaluation file not found at {eval_b_path}. Please run Stage B evaluation first.")

### 📊 Stage C: Stance and Relevance Classification Performance
We evaluate the model's accuracy on determining if a passage is relevant, and whether it assigns the correct stance direction using `macro-F1` and `quote accuracy` (verbatim correctness).

In [ ]:
from sklearn.metrics import f1_score, classification_report

# 1. Gather all annotated evaluation rows from the cache
recs_eval = []
for k in EVAL_PAIRS:
    # Get Qwen3.8-27B Czech judgements
    rec = cache.get(hashlib.sha1(json.dumps([LLM_MODEL, PROMPT_VERSION, "cs", k[0], k[1], PARAMS[LLM_MODEL]], sort_keys=True).encode()).hexdigest())
    if rec:
        recs_eval.append(rec)

df_eval = pd.DataFrame(recs_eval)

# 2. Merge with gold targets
gold_series = G.rename(columns={"gold": "gold_stance"})
df_eval = df_eval.merge(gold_series, on=["thesis_id", "passage_id"], how="inner")

# Map gold stance values to relevance and direction classes
df_eval["gold_relevant"] = df_eval["gold_stance"].isin([1, -1])

# Predictions
df_eval["pred_relevant"] = df_eval["relevant"].fillna(False)
df_eval["pred_stance"] = df_eval["stance"].fillna(0.0)

# 3. Print Overall Evaluation Metrics by Split
for split_name in ["dev", "test"]:
    split_df = df_eval[df_eval["split"] == split_name]
    if split_df.empty:
        continue

    print(f"=== SPLIT: {split_name.upper()} ({len(split_df)} pairs) ===")

    # Relevance F1
    f1_rel = f1_score(split_df["gold_relevant"], split_df["pred_relevant"], average="macro")
    print(f"Relevance Macro-F1: {f1_rel:.3f}")

    # Stance F1 (only on agreed/evaluated instances, map -1, 0, 1)
    y_true_stance = split_df["gold_stance"].astype(int)
    y_pred_stance = split_df["pred_stance"].map(lambda x: int(np.sign(x)) if pd.notna(x) else 0)
    f1_st = f1_score(y_true_stance, y_pred_stance, average="macro")
    print(f"Stance Direction Macro-F1: {f1_st:.3f}")

    # Quote Accuracy (verbatim match in original text)
    quote_acc = split_df[split_df["relevant"] == True]["quote_ok"].mean()
    print(f"Quote Accuracy on Relevant Passages: {quote_acc:.3%}\n")


### 📊 Comprehensive Evaluation & Visualization (Combined Gold Standard & Hard Negatives)
We will now merge the positive gold standard evaluation dataset with the agreed hard negatives from `stance_negatives_v5.csv` and run a comprehensive performance test. This block will output detailed statistics and visualize the results using confusion matrices for both the **DEV** and **TEST** splits.

# Evaluation
New claims (q12–q36, all test). Retrieval uses the app's function (`per_party(dense_scores(claim), k=K_STANCE)`).
Pairs are chosen without any judge output. Two annotators label every pair independently, and the labels and thresholds are frozen before the judge runs.

In [ ]:
# E1 | Old data report: A-vs-B kappa + disputed pairs, gold inside top-5 (all 12), estimated share of relevant among the 50 retrieved
from sklearn.metrics import cohen_kappa_score
neg = pd.read_csv(NEG_PATH, encoding="utf-8-sig")
print("raw label values:", {c: neg[c].value_counts(dropna=False).to_dict() for c in ("label_A", "label_B")})
def norm_label(x):
    s = re.sub(r"[\s_]+", " ", str(x)).strip().upper()
    return "NOT_RELEVANT" if s.startswith("NOT REL") or s in ("NO POSITION", "NAN", "") else s
for c in ("label_A", "label_B"): neg[c + "_n"] = neg[c].map(norm_label)
print("normalised:", {c: neg[c + "_n"].value_counts().to_dict() for c in ("label_A", "label_B")})
print(f"old negatives: kappa A vs B {cohen_kappa_score(neg['label_A_n'], neg['label_B_n']):.3f} | raw agreement "
      f"{(neg['label_A_n'] == neg['label_B_n']).sum()}/{len(neg)} | rows dropped by C6's exact-string filter: {(neg['label_A'] != neg['label_B']).sum()}")
print(neg.loc[neg["label_A_n"] != neg["label_B_n"], ["thesis_id", "passage_id", "label_A_n", "label_B_n"]].to_string())

top5 = ret[ret["rank_in_party"] <= K_STANCE]
in5 = set(zip(top5["thesis_id"], top5["passage_id"]))
gp = pd.DataFrame([(g.query_id, g.split, p) for _, g in Q.iterrows() for p in g.rel], columns=["thesis_id", "split", "passage_id"])
gp["in_top5"] = [k in in5 for k in zip(gp["thesis_id"], gp["passage_id"])]
print("\ngold passages inside the party's top-5:", gp.groupby("split")["in_top5"].agg(["sum", "size"]).to_dict("index"),
      f"| all 12 claims: {gp['in_top5'].sum()}/{len(gp)}")

cons = neg[neg["label_A_n"] == neg["label_B_n"]]                     # human consensus only
est = []
for q, g in top5.groupby("thesis_id"):
    gold_in = gp[(gp["thesis_id"] == q) & gp["in_top5"]].shape[0]
    c = cons[cons["thesis_id"] == q]
    p_rel = (c["label_A_n"] != "NOT_RELEVANT").mean() if len(c) else np.nan
    est.append(dict(thesis_id=q, retrieved=len(g), gold_in_top5=gold_in, neg_labelled=len(c),
                    est_relevant=gold_in + (len(g) - gold_in) * p_rel))
est = pd.DataFrame(est); est["share"] = est["est_relevant"] / est["retrieved"]
print("\nestimated share of relevant passages among the 50 retrieved (gold + consensus-labelled random negatives):")
print(est.round(2).to_string(index=False), f"\nmean share {est['share'].mean():.2f} (per claim from only ~5 labelled negatives: rough)")

In [ ]:
# E2 | Ondra's claims (eval2_claims.csv in CORPUS_DIR, unchanged): validate, freeze by SHA-256 on first run, slices, closeness to the old 12
CLAIMS_PATH = os.path.join(CORPUS_DIR, "eval2_claims.csv"); HASH_PATH = CLAIMS_PATH + ".sha256"
assert os.path.exists(CLAIMS_PATH), "put Ondra's eval2_claims.csv into CORPUS_DIR first (do not edit it after that)"
digest = hashlib.sha256(open(CLAIMS_PATH, "rb").read()).hexdigest()
if not os.path.exists(HASH_PATH): open(HASH_PATH, "w").write(digest); print("frozen now:", digest[:16])
assert digest == open(HASH_PATH).read().strip(), "eval2_claims.csv changed after freezing"
cl = pd.read_csv(CLAIMS_PATH, encoding="utf-8-sig").rename(columns={"query_id": "claim_id", "query": "claim"})
assert len(cl) == 25 and cl["claim_id"].is_unique and cl["claim"].is_unique and (cl["split"] == "test").all()
assert not set(cl["claim_id"]) & set(Q["query_id"]) and not set(cl["claim"]) & set(Q["query"]), "id or text repeats an old claim"
assert cl["claim"].str.contains("[ěščřžýáíéůú]").mean() > 0.8, "diacritics lost on read"

TAGS = {  # slices for the report; from the wording only. Ondra adds "likely_reject" (>= 8 claims) BEFORE the first run
    "two_part": ["q14", "q15", "q21", "q22", "q33"],
    "ban":      ["q13", "q18", "q20", "q22", "q27"],
    "likely_reject": [],
}
TAGS_PATH = os.path.join(CORPUS_DIR, "eval2_claim_tags.json")
if not os.path.exists(TAGS_PATH): json.dump(TAGS, open(TAGS_PATH, "w"), indent=1)
TAGS = json.load(open(TAGS_PATH))
print({t: len(v) for t, v in TAGS.items()}, "| claims in no slice:", sorted(set(cl["claim_id"]) - {c for v in TAGS.values() for c in v}))

def qvec(q):                                       # the retriever's own query vector (cached in B3), not the judge
    dense_scores(q); return qcache[hashlib.sha1(f"{MODEL}|{INSTR}|{q}".encode("utf-8")).hexdigest()]
sim = np.array([qvec(q) for q in cl["claim"]]) @ np.array([qvec(q) for q in Q["query"]]).T
near = sorted(((sim[i, j], cl["claim_id"][i], Q["query_id"].iloc[j]) for i in range(len(cl)) for j in range(len(Q))), reverse=True)[:6]
print("closest new/old claim pairs (a human decides whether any is a paraphrase):")
for s, a, b in near: print(f"  {s:.2f} {a} {cl.set_index('claim_id').loc[a, 'claim']}  <->  {b} {Q.set_index('query_id').loc[b, 'query']}")

In [ ]:
# E2b | "likely_reject" slice = claims with at least one explicit opposing passage (keyword check of the programmes, F027); set BEFORE any judge result
TAGS["likely_reject"] = ["q13", "q14", "q16", "q17", "q19", "q22", "q35", "q36"]
json.dump(TAGS, open(TAGS_PATH, "w"), indent=1)
print({t: len(v) for t, v in TAGS.items()})

In [ ]:
# E3 | Retrieve with the app's function, sample blocks A/B/C (no judge output used), print the checks
assert hashlib.sha256(open(CLAIMS_PATH, "rb").read()).hexdigest() == open(HASH_PATH).read().strip(), "claims not frozen"
rng = np.random.default_rng(SEED)
pool = pd.concat([per_party(dense_scores(r.claim), k=K_STANCE).assign(claim_id=r.claim_id, claim=r.claim) for r in cl.itertuples()],
                 ignore_index=True)                                    # exactly what the app sends to the judge
pool["text"] = pool["passage_id"].map(TEXT)
CUES = {  # Czech cue lexicon (block C); matched on lower-cased passage text
    "rejection":   r"\b(?:odmít|nesouhlas|nepodpoř|zastavíme|zrušíme|nebudeme|jsme proti|nesmí|nepřipustíme)",
    "conditional": r"\b(?:prověř|zváž|posoudíme|pokud)",
    "support":     r"\b(?:prosadíme|zajistíme|podpoříme|zavedeme|postavíme|zahájíme)",
    "hedged":      r"\b(?:budeme usilovat|pokusíme|postupně|v rámci možností|chceme)",
    "other_party": r"(?:současn\w* koalic|předchozí vedení|minulé vedení|bývalé|politici)",
}
for g, rx in CUES.items(): pool["cue_" + g] = pool["text"].str.lower().str.contains(rx, regex=True)

picked = []
for cid, g in pool.groupby("claim_id", sort=True):                    # A: 2 random from ranks 1-2, 2 from ranks 3-5
    for lo, hi in ((1, 2), (3, 5)):
        s = g[g["rank_in_party"].between(lo, hi)]
        take = s.loc[rng.choice(s.index, 2, replace=False)]
        picked.append(take.assign(block="A", stratum=f"r{lo}-{hi}", incl_prob=2 / len(s)))
A = pd.concat(picked)
B = pd.concat([g.drop(index=A.index, errors="ignore").nlargest(2, "score").assign(block="B", stratum="top_score", incl_prob=np.nan)
               for _, g in pool.groupby("claim_id", sort=True)])
used, Cpick, cnt = set(A.index) | set(B.index), [], dict.fromkeys(CUES, 0)
for cid, g in pool.groupby("claim_id", sort=True):                    # C: 2 per claim, always the cue group with the fewest picks so far
    g = g.drop(index=list(used & set(g.index)))
    for _ in range(2):
        avail = [c for c in CUES if g["cue_" + c].any()]
        if not avail: break
        grp = min(avail, key=lambda c: (cnt[c], list(CUES).index(c)))
        i = rng.choice(g.index[g["cue_" + grp]]); cnt[grp] += 1
        Cpick.append(g.loc[[i]].assign(block="C", stratum="cue_" + grp, incl_prob=np.nan)); g = g.drop(index=i)
C_ = pd.concat(Cpick)
M = pd.concat([A, B, C_]).reset_index(drop=True)
M["pair_id"] = [f"p{i:03d}" for i in rng.permutation(len(M))]          # random ids: the id says nothing about block or rank
M["party"] = M["passage_id"].map(dict(zip(df_clust["passage_id"], df_clust["party"])))

old_pairs = {(g.query_id, p) for _, g in Q.iterrows() for p in g.rel} | set(zip(neg["thesis_id"], neg["passage_id"]))
pc = M["party"].value_counts()
print("blocks:", M["block"].value_counts().to_dict(), "| cue groups in C:", cnt)
print("CHECK no duplicate pairs:", not M.duplicated(["claim_id", "passage_id"]).any())
print("CHECK no overlap with old gold/negative pairs:", not (set(zip(M["claim_id"], M["passage_id"])) & old_pairs),
      f"| passages also used in old pairs (info): {M['passage_id'].isin({p for _, p in old_pairs}).sum()}")
print("CHECK <= 10 pairs per claim:", M.groupby("claim_id").size().max() <= 10)
print("CHECK every party >= 10 pairs:", (pc >= 10).all(), "| none above 20 %:", (pc / len(M) <= 0.20).all())
print(pc.to_dict())
print("CHECK every cue group >= 8 in block C:", {g: n >= 8 for g, n in cnt.items()})

In [ ]:
# E4 | Master file + two differently shuffled annotation sheets + 15-pair calibration sheet; never overwritten, read-back checked
MASTER = os.path.join(CORPUS_DIR, "eval2_pairs_master.csv")
SHEETS = {"A": os.path.join(CORPUS_DIR, "eval2_sheet_A.csv"), "B": os.path.join(CORPUS_DIR, "eval2_sheet_B.csv")}
CALIB = os.path.join(CORPUS_DIR, "eval2_calibration.csv")
COLS = ["pair_id", "claim", "passage", "relevant", "direction", "hedged", "note"]   # relevant Y/N | direction AGREE/DISAGREE/MIXED | hedged Y/N
def write_once(df_, path):
    if os.path.exists(path): print("exists, NOT overwritten:", os.path.basename(path)); return
    df_.to_csv(path, index=False, encoding="utf-8-sig")
    back = pd.read_csv(path, encoding="utf-8-sig")
    txt = "passage" if "passage" in back else "text"
    assert back[txt].tolist() == df_[txt].tolist(), "text changed on read-back"
    assert back[txt].str.contains("[ěščřžýáíéůú]").mean() > 0.9, "diacritics lost on read-back"
    print(f"{os.path.basename(path)}: {len(back)} rows, read-back OK")
write_once(M.drop(columns=[c for c in M if c.startswith("cue_")]).assign(cues=M[[c for c in M if c.startswith("cue_")]].apply(
    lambda r: ",".join(c[4:] for c, v in r.items() if v), axis=1)), MASTER)
sheet = M[["pair_id", "claim", "text"]].rename(columns={"text": "passage"}).assign(relevant="", direction="", hedged="", note="")[COLS]
for k, (name, path) in enumerate(SHEETS.items(), 1):                # same rows, different order per annotator
    write_once(sheet.sample(frac=1, random_state=SEED + k).reset_index(drop=True), path)
rest = pool.drop(index=pool.index[pool.set_index(["claim_id", "passage_id"]).index.isin(M.set_index(["claim_id", "passage_id"]).index)])
cue_any = rest[[c for c in rest if c.startswith("cue_")]].any(axis=1)
calib = pd.concat([rest[cue_any].sample(5, random_state=SEED), rest[~cue_any].sample(10, random_state=SEED)])
write_once(calib.assign(pair_id=[f"cal{i:02d}" for i in range(len(calib))])[["pair_id", "claim", "text"]]
           .rename(columns={"text": "passage"}).assign(relevant="", direction="", hedged="", note="")[COLS], CALIB)

In [ ]:
# E5 | Validate both sheets (+ top-up if any), kappa, consensus + disputed stratum, freeze eval2_labels_frozen.csv with SHA-256
SHEETS = {"A": os.path.join(CORPUS_DIR, "eval2_sheet_A.csv"), "B": os.path.join(CORPUS_DIR, "eval2_sheet_B.csv")}
LABELS_PATH = os.path.join(CORPUS_DIR, "eval2_labels_frozen.csv"); LABELS_HASH = LABELS_PATH + ".sha256"
YN = {"Y": "Y", "YES": "Y", "A": "Y", "ANO": "Y", "N": "N", "NO": "N", "NE": "N"}
DIRS = {"AGREE", "DISAGREE", "MIXED"}
def clean(v): return "" if pd.isna(v) else re.sub(r"\s+", " ", str(v)).strip().upper()
TOPUP_MASTER = os.path.join(CORPUS_DIR, "eval2_topup_master.csv")
TOPUP_SHEETS = {w: os.path.join(CORPUS_DIR, f"eval2_topup_sheet_{w}.csv") for w in "AB"}
def read_all(paths): return pd.concat([pd.read_csv(p, encoding="utf-8-sig", dtype=str) for p in paths if os.path.exists(p)], ignore_index=True)
lab, bad = {}, []
for who, path in SHEETS.items():
    s = read_all([path, TOPUP_SHEETS[who]])                                     # main sheet + top-up sheet if it exists
    print(who, "raw values:", {c: s[c].map(clean).value_counts().to_dict() for c in ("relevant", "direction", "hedged")})
    s["rel"] = s["relevant"].map(clean).map(YN); s["dir"] = s["direction"].map(clean); s["hed"] = s["hedged"].map(clean).map(YN)
    for r in s.itertuples():
        ok = (r.rel == "N" and r.dir == "" and pd.isna(r.hed)) or \
             (r.rel == "Y" and r.dir == "MIXED" and pd.isna(r.hed)) or \
             (r.rel == "Y" and r.dir in DIRS - {"MIXED"} and r.hed in ("Y", "N"))
        if not ok: bad.append((who, r.pair_id, r.relevant, r.direction, r.hedged))
    lab[who] = s.set_index("pair_id")
if bad:
    print(f"{len(bad)} rows break the rules (direction only if relevant=Y; hedged only with AGREE/DISAGREE):")
    print(pd.DataFrame(bad, columns=["sheet", "pair_id", "relevant", "direction", "hedged"]).to_string(index=False))
assert not bad, "fix these rows in the sheets, then rerun E5"
L = lab["A"][["rel", "dir", "hed"]].join(lab["B"][["rel", "dir", "hed"]], lsuffix="_A", rsuffix="_B", how="outer")
master = read_all([os.path.join(CORPUS_DIR, "eval2_pairs_master.csv"), TOPUP_MASTER]).set_index("pair_id")
assert len(L) == len(master) and L.notna()[["rel_A", "rel_B"]].all().all(), "A and B sheets do not cover the same pair_ids as the master"
for w in "AB": L["lab4_" + w] = np.where(L["rel_" + w] == "N", "NOT_RELEVANT", L["dir_" + w])
L["disputed"] = L["lab4_A"] != L["lab4_B"]
L["consensus"] = np.where(L["disputed"], "", L["lab4_A"])
both = L[(L["rel_A"] == "Y") & (L["rel_B"] == "Y")]
print(f"kappa relevance {cohen_kappa_score(L['rel_A'], L['rel_B']):.3f} (n={len(L)}) | kappa direction on both-relevant "
      f"{cohen_kappa_score(both['dir_A'], both['dir_B']):.3f} (n={len(both)}) | kappa 4-class {cohen_kappa_score(L['lab4_A'], L['lab4_B']):.3f}")
print("consensus:", L.loc[~L["disputed"], "consensus"].value_counts().to_dict(), f"| disputed {L['disputed'].sum()}/{len(L)}")
n_dis = (L["consensus"] == "DISAGREE").sum()
topped = os.path.exists(TOPUP_MASTER)
print(f"consensus DISAGREE: {n_dis}", "-> run the top-up wave (E5b), label it, rerun E5" if n_dis < 20 and not topped else "")
out = master[["claim_id", "passage_id", "party", "block", "stratum", "incl_prob"]].join(L, how="inner").rename_axis("pair_id").reset_index()
FREEZE = n_dis >= 20 or topped                      # one top-up wave at most, then freeze whatever the count
if FREEZE and not os.path.exists(LABELS_PATH):
    out.to_csv(LABELS_PATH, index=False, encoding="utf-8-sig")
    open(LABELS_HASH, "w").write(hashlib.sha256(open(LABELS_PATH, "rb").read()).hexdigest()); print("labels frozen:", LABELS_PATH)
elif os.path.exists(LABELS_PATH): print("exists, NOT overwritten:", LABELS_PATH)

In [ ]:
if os.path.exists(os.path.join(CORPUS_DIR, "eval2_labels_frozen.csv.sha256")):
    # E6 | Pre-registration BEFORE any judge call on eval2: thresholds + headline metrics + the exact judge config, frozen
    PREREG_PATH = os.path.join(CORPUS_DIR, "eval2_prereg.json")
    THRESHOLDS = {  # B fills ALL of these before any judge call on eval2; the verdict is read against them
        "valid_json_min": None, "relevance_recall_min": None, "fpr_blockA_max": None,
        "disagree_recall_min": None, "max_agree_disagree_flips": None, "judge_vs_consensus_kappa_min": None,
    }
    HEADLINE = ["relevance recall", "block-A FPR (weighted)", "DISAGREE recall", "AGREE<->DISAGREE flips"]   # everything else = exploratory
    cfg = dict(model=LLM_MODEL, model_2=LLM_MODEL_2, params=PARAMS[LLM_MODEL], params_2=PARAMS[LLM_MODEL_2], prompt_version=PROMPT_VERSION,
               prompt_sha=hashlib.sha256(json.dumps(PROMPTS, ensure_ascii=False, sort_keys=True).encode()).hexdigest(), tau=TAU, k_stance=K_STANCE,
               claims_sha=open(HASH_PATH).read().strip(), labels_sha=open(LABELS_HASH).read().strip(),
               tags_sha=hashlib.sha256(open(TAGS_PATH, "rb").read()).hexdigest())
    if not os.path.exists(PREREG_PATH):
        assert all(v is not None for v in THRESHOLDS.values()), "fill every threshold first"
        json.dump(dict(thresholds=THRESHOLDS, headline=HEADLINE, config=cfg, written=time.strftime("%Y-%m-%d %H:%M")),
                  open(PREREG_PATH, "w"), indent=1, ensure_ascii=False)
        print("pre-registered:", hashlib.sha256(open(PREREG_PATH, "rb").read()).hexdigest()[:16])
    pre = json.load(open(PREREG_PATH))
    changed = {k: (pre["config"][k], v) for k, v in cfg.items() if pre["config"][k] != v}
    assert not changed, f"judge config differs from the pre-registration: {changed}"
    print("pre-registration OK; config unchanged:", {k: cfg[k] for k in ("model", "prompt_version", "tau")})
else:
    print('E6 skipped: eval2 labels are not frozen yet (finish E5 + top-up first). The app does not need E6.')


# App adapter

In [ ]:
# D1 | App adapter: Stage B (dense, top-K per party) + Stage C (prompt v3, parallel calls) for one free-text claim
#      Same model, params, prompt, K and frozen tau as evaluated in C6. score in [-1, 1]; "confidence" is an extra field.
import json, hashlib, threading
from concurrent.futures import ThreadPoolExecutor

assert PROMPT_VERSION == "v3" and "passage_score" in globals(), "run C2 (prompt v3) and C6 first"
APP_MODEL  = LLM_MODEL                                        # judge chosen in C3 (Qwen3.8-27B)
APP_PARAMS = PARAMS[APP_MODEL]                                # same params as evaluated
WORKERS    = 10                                               # parallel calls; lower it if the proxy answers 429 / timeouts
TAU        = json.load(open(TAU_PATH))["tau"]                 # frozen in C6, never changed here
APP_CACHE  = os.path.join(CORPUS_DIR, "app_stance_cache_v5.jsonl")   # separate from the evaluation cache
FIX_CS = "Tvoje odpověď nebyla platná ({err}). Odpověz znovu pouze jedním objektem JSON podle pokynů; citace musí být doslovně zkopírovaná z úryvku."

TEXT = dict(zip(df_clust["passage_id"], df_clust["text"]))
PAGE = dict(zip(df_clust["passage_id"], df_clust["page"]))
PARTIES = sorted(df_clust["party"].unique())

app_cache, lock = {}, threading.Lock()
if os.path.exists(APP_CACHE):
    with open(APP_CACHE, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line); app_cache[r["key"]] = r

LAST_ERROR = {"msg": ""}                              # last LLM error, shown by T1 and in the page's error box

def ask_msgs(msgs):                                   # network errors: 3 tries, then None (the error is logged, not hidden)
    for t in range(3):
        try:
            return client.with_options(timeout=180).chat.completions.create(
                model=APP_MODEL, messages=msgs, **APP_PARAMS).choices[0].message.content or ""
        except Exception as e:
            LAST_ERROR["msg"] = f"{type(e).__name__}: {str(e)[:200]}"
            print("LLM call failed:", LAST_ERROR["msg"])
            time.sleep(5 * (t + 1))
    return None

def judge_claim(claim, pid):                          # same rule as C4: one re-ask with the reason
    key = hashlib.sha1(json.dumps([APP_MODEL, PROMPT_VERSION, "cs", claim, pid, APP_PARAMS], sort_keys=True).encode()).hexdigest()
    if key in app_cache:
        return app_cache[key]
    passage = TEXT[pid]
    msgs = [{"role": "user", "content": make_prompt("cs", claim, passage)}]
    for attempt in (1, 2):
        raw = ask_msgs(msgs)
        if raw is None:
            return {"relevant": None, "net_error": True}   # network failure: not cached, counts as no position
        d, err = parse(raw, passage)
        if d is not None and d["quote_ok"]:
            break
        msgs += [{"role": "assistant", "content": raw}, {"role": "user", "content": FIX_CS.format(err=err or "quote not verbatim")}]
    rec = dict(key=key, claim=claim, passage_id=pid,
               relevant=None if d is None else d["relevant"], stance=None if d is None else d["stance"],
               confidence=None if d is None else d["confidence"],
               quote="" if d is None else d["quote"], quote_ok=bool(d) and d["quote_ok"])
    with lock:                                        # threads share the cache and the file
        app_cache[key] = rec
        with open(APP_CACHE, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return rec

def score_claim(claim: str) -> list[dict]:
    claim = claim.strip()
    hits = per_party(dense_scores(claim), k=K_STANCE)                     # 10 parties x 5 = 50 passages
    with ThreadPoolExecutor(WORKERS) as pool:                             # 50 judge calls, WORKERS at a time
        recs = list(pool.map(lambda pid: judge_claim(claim, pid), hits["passage_id"]))
    failed = sum(bool(r.get("net_error")) for r in recs)
    if failed == len(recs):                                               # nothing reached the model: say so in the page
        raise RuntimeError("Model LLM není dostupný (" + LAST_ERROR["msg"] + "). Zkuste to znovu později.")
    if failed: print(f"WARNING: {failed}/{len(recs)} judge calls failed and count as no position:", LAST_ERROR["msg"])
    rows = []
    for h, r in zip(hits.itertuples(), recs):
        s = passage_score(r, TAU)                                         # C6 rule: c * stance, None below tau
        if s is None: continue                                            # NO POSITION: not sent, counts as 0
        page = PAGE[h.passage_id]
        rows.append(dict(party=h.party, quote=r["quote"], page=None if pd.isna(page) else int(page),
                         score=round(s, 3), confidence=r["confidence"]))
    return rows

# quick test: the first run calls the model, the second must come from the cache (about 0 s)
for run in (1, 2):
    t0 = time.time(); test = score_claim("Praha by měla dostavět metro D.")
    print(f"run {run}: {time.time() - t0:.0f} s | {len(test)} rows")
print(pd.DataFrame(test).head(10).to_string())

def rank_parties(queries):
    """Server-side ranking = formula (3) with the neutral tie rule. index.html uses it instead of its own aggregate()."""
    out = []
    for p in PARTIES:
        per, flags = [], []
        for q in queries:
            s, flag = claim_score([r["score"] for r in q["results"] if r["party"] == p])
            per.append(s); flags.append(flag)
        out.append(dict(party=p, score=sum(per) / len(per), perClaim=per, flags=flags,
                        coverage=sum(f != "none" for f in flags)))      # documented claims, contradictions included
    return sorted(out, key=lambda r: (-r["score"], -r["coverage"], r["party"]))

In [ ]:
# D2 | Start the app server in this session (index.html + server.py in <PROJECT_DIR>/app)
import sys, socket, threading, importlib
APP_DIR = str(PROJECT_DIR / "app")
assert os.path.isfile(os.path.join(APP_DIR, "server.py")), f"server.py not found in {APP_DIR}"
if APP_DIR not in sys.path: sys.path.insert(0, APP_DIR)
importlib.invalidate_caches()                      # Drive files added after the session started
import server
print("server module:", server.__file__)

server.score_claim = score_claim                   # plug the notebook pipeline into the server
server.rank_parties = rank_parties                 # server-side ranking with contradiction flags (D1)
server.PARTIES = PARTIES
server.TAU = TAU
server.PROMPT_VERSION = PROMPT_VERSION
server.MODEL_NAME = f"{APP_MODEL} · prompt {PROMPT_VERSION} · corpus {P['corpus_version']}"

PORT = 5000
with socket.socket() as s:
    busy = s.connect_ex(("127.0.0.1", PORT)) == 0
if busy:
    print(f"Port {PORT} is already in use. If T1 shows 'TEST' or old values, use Runtime -> Restart session, then rerun.")
else:
    threading.Thread(target=lambda: server.app.run(port=PORT, use_reloader=False), daemon=True).start()
    time.sleep(1)

from google.colab.output import eval_js
print(eval_js(f"google.colab.kernel.proxyPort({PORT})"))     # open this link


In [ ]:
# T1 | Smoke test of the running app: health, one live (uncached) LLM call, one analyze request. Read all three lines.
import requests
base = f"http://127.0.0.1:{PORT}"
h = requests.get(base + "/api/health", timeout=10)
print("1 health :", h.status_code, h.text[:300])
if "TEST" in h.text: print("  -> this is server_test.py, not the real server: Runtime -> Restart session, rerun D1 + D2")

raw = ask_msgs([{"role": "user", "content": 'Odpověz pouze tímto JSON a ničím jiným: {"ok": true}'}])   # never cached
print("2 LLM    :", "FAILED -> " + LAST_ERROR["msg"] if raw is None else "OK " + repr(raw.strip()[:60]))

r = requests.post(base + "/api/analyze", json={"claims": ["Praha by měla dostavět metro D."]}, timeout=900)
print("3 analyze:", r.status_code, r.text[:400])


## Export for the web app
One cell, added for the GitHub + Render migration. It writes `app/data/*` and
`tests/fixtures/*` from this run, so the app and the notebook cannot drift apart.
It makes no API call and changes no method. Run it after `D1`.

In [ ]:
# E0 | Export for the web app: writes app/data/* and tests/fixtures/* for the GitHub + Render repo.
#      Run AFTER D1 (needs score_claim, app_cache, qcache) so every value comes from this run and
#      nothing is retyped by hand. This cell changes no method: it only copies what C2/C6/D1 decided.
#      Makes no API call: the parity claims are taken from answers that are already cached.
import csv
import hashlib
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

assert "score_claim" in globals(), "run D1 first"
assert PROMPT_VERSION == "v3", f"expected prompt v3, got {PROMPT_VERSION}"

EXPORT_DIR = Path(PROJECT_DIR) / "export"      # mirrors the repo layout, so the team copies two folders across
assert EXPORT_DIR.name == "export", "refusing to clear anything but the export folder"
DATA_DIR, FIX_DIR = EXPORT_DIR / "app" / "data", EXPORT_DIR / "tests" / "fixtures"
shutil.rmtree(EXPORT_DIR, ignore_errors=True)  # a stale file must never survive a re-export
DATA_DIR.mkdir(parents=True)
FIX_DIR.mkdir(parents=True)

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def query_key(claim):                          # exactly B3's key, so the app reuses the same cache entry
    return hashlib.sha1(f"{MODEL}|{INSTR}|{claim}".encode("utf-8")).hexdigest()

def judge_key(claim, pid):                     # exactly D1's key
    return hashlib.sha1(json.dumps([APP_MODEL, PROMPT_VERSION, "cs", claim, pid, APP_PARAMS],
                                   sort_keys=True).encode()).hexdigest()

# ---- 1. passages.csv: the 696 non-noise passages, in df_clust order (= E_qwen row order) ----
# Only the four columns the app needs. page is left empty when NaN: TXT/JSON sources have no page.
passages_csv = DATA_DIR / "passages.csv"
with open(passages_csv, "w", encoding="utf-8", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["passage_id", "party", "page", "text"])
    for r in df_clust.itertuples():
        w.writerow([r.passage_id, r.party, "" if pd.isna(r.page) else int(r.page), r.text])

# ---- 2. passage_emb.npy: E_qwen, float32, same row order as passages.csv ----
emb_npy = DATA_DIR / "passage_emb.npy"
E_export = np.ascontiguousarray(E_qwen, dtype=np.float32)
assert E_export.shape[0] == len(df_clust), f"{E_export.shape[0]} vectors vs {len(df_clust)} passages"
np.save(emb_npy, E_export)

# ---- 3. app_config.json: the single source of truth the app reads instead of retyping anything ----
# APP_MODEL, never P["llm_model"]: C1 sets P["llm_model"] only as a starting value, and C3 then
# overwrites LLM_MODEL with the smoke-test winner. APP_MODEL is the judge that was actually evaluated.
config = {
    "embed_model": str(MODEL),
    "llm_model": str(APP_MODEL),
    "llm_params": APP_PARAMS,
    "prompt_version": PROMPT_VERSION,
    "prompt_cs": PROMPTS["cs"],
    "instr": INSTR,
    "k_stance": int(K_STANCE),
    "tau": TAU,
    "corpus_version": P["corpus_version"],
    "n_passages": len(df_clust),
    "emb_dim": int(E_export.shape[1]),
    "parties": sorted(str(p) for p in df_clust["party"].unique()),
    "exported_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "sha256_passages_csv": sha256(passages_csv),
    "sha256_passage_emb_npy": sha256(emb_npy),
}
(DATA_DIR / "app_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- 4. Parity fixtures: 3 claims whose answers are ALREADY cached, so this makes no API call ----
# The repo's test loads these into pipeline.py's two caches and uses a fake LLM client that fails if
# it is ever called. A wrong model/prompt/params/prefix gives a different cache key, so the fake
# client gets called and the test fails. Wrong retrieval or scoring gives different rows instead.
PREFERRED = "Praha by měla dostavět metro D."          # D1's own test claim, so it is always cached
cached_claims = sorted({r["claim"] for r in app_cache.values()})
candidates = ([PREFERRED] if PREFERRED in cached_claims else []) + \
             [c for c in cached_claims if c != PREFERRED]
print(f"{len(cached_claims)} distinct claims in app_cache")

def fully_cached(claim):
    """(ok, qkey, judge_keys) - ok only when the query vector AND every judge answer are cached."""
    qkey = query_key(claim)
    if qkey not in qcache:
        return False, qkey, []
    hits = per_party(dense_scores(claim), k=K_STANCE)   # served from qcache, so no API call
    keys = [judge_key(claim, pid) for pid in hits["passage_id"]]
    return all(k in app_cache for k in keys), qkey, keys

chosen, vectors, skipped = [], {}, []
for claim in candidates:
    if len(chosen) == 3:
        break
    ok, qkey, keys = fully_cached(claim)
    if not ok:
        skipped.append(claim)
        continue
    rows = score_claim(claim)                           # cache-served, no network
    chosen.append({"claim": claim, "query_cache_key": qkey,
                   "judge_answers": [app_cache[k] for k in keys],
                   "expected_rows": rows})
    vectors[qkey] = np.asarray(qcache[qkey], dtype=np.float32)
    neg = sum(r["score"] < 0 for r in rows)
    print(f"picked: {claim[:58]!r} | {len(rows)} rows | {neg} negative")

if len(chosen) < 3:
    print(f"\nOnly {len(chosen)} of the 3 parity claims are fully cached "
          f"({len(skipped)} candidates were incomplete).")
    print("D1's self-test caches one claim only. To get three, run score_claim once per extra claim")
    print("in a scratch cell (about 50 judge calls each), then re-run this cell. For example:")
    print('    for c in ["Praha má zdražit jízdné v MHD.",')
    print('              "Praha má postavit více obecních bytů."]:')
    print("        score_claim(c)")
    raise AssertionError(f"need 3 fully cached claims, have {len(chosen)}")

if not any(r["score"] < 0 for c in chosen for r in c["expected_rows"]):
    print("NOTE: no negative score among the chosen claims. Parity still holds but covers less;")
    print("      prefer a claim where some party disagrees, so the sign handling is exercised.")

(FIX_DIR / "parity_expected.json").write_text(
    json.dumps({"config": config, "claims": chosen}, ensure_ascii=False, indent=2), encoding="utf-8")
# Query vectors go to .npz rather than JSON: float32 round-trips exactly, and the JSON stays readable.
np.savez(FIX_DIR / "parity_query_vectors.npz", **vectors)

# ---- 4b. Warm start: every cached answer, so the app is fast after a spin-down ----
# Render's filesystem is ephemeral and a free service spins down after 15 idle minutes, so the
# app's in-memory caches are empty again on every restart. Without these two files the first
# analysis of a session costs the spin-up plus k_stance x parties live judge calls per claim.
# The app looks these up under the keys IT computes, so a stale file is simply ignored, never
# wrong: a changed model, prompt or params gives different keys and the app calls the model.
warm_jsonl = DATA_DIR / "app_stance_cache.jsonl"
with open(warm_jsonl, "w", encoding="utf-8", newline="\n") as fh:
    for rec in app_cache.values():
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

# Query vectors as JSON, not .npz: float32 survives the round trip exactly once the app casts
# back to float32 on load, and a readable file is easier for the team to inspect. The parity
# fixtures use .npz because there the file is test data nobody reads by hand.
warm_q = {}
for claim in sorted({r["claim"] for r in app_cache.values()}):
    k = query_key(claim)
    if k in qcache:
        warm_q[k] = [float(x) for x in np.asarray(qcache[k], dtype=np.float32)]
(DATA_DIR / "query_cache.json").write_text(
    json.dumps(warm_q, ensure_ascii=False), encoding="utf-8")
print(f"\nwarm start: {len(app_cache)} judge answers, {len(warm_q)} query vectors")

# ---- 5. Report ----
print("\nexported to", EXPORT_DIR)
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(EXPORT_DIR).as_posix():42} {p.stat().st_size:>11,} B")
print(f"\njudge: {APP_MODEL} {APP_PARAMS} | tau: {TAU} | k_stance: {K_STANCE}"
      f" | passages: {len(df_clust)} | dim: {config['emb_dim']}")
print("Copy export/app/data -> repo app/data, and export/tests/fixtures -> repo tests/fixtures")
